In [ ]:
import os
import pandas as pd
import os
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import cv2

In [ ]:
#Atualizar para usar no collab

!pip install --upgrade pandas

Parte 1 - EDA

In [ ]:
pklfiles = [f for f in os.listdir('Project_Features') if os.path.isfile(os.path.join('Project_Features', f)) and f.endswith('_speech.pkl')]
print(pklfiles)

In [ ]:
# Load one
for speech in pklfiles:
    print(speech)
    data = pd.read_pickle(os.path.join('Project_Features', speech))

# Print first lines
print(data.head())

In [ ]:
import pandas as pd
import os

# 1. Definir o caminho para o ficheiro de speech
pasta = "Project_Features"
nome_ficheiro_speech = "Ventura_vs_Seguro_November_17_speech.pkl"
caminho_completo = os.path.join(pasta, nome_ficheiro_speech)

print(f"A carregar as transcrições de: {caminho_completo}...")

try:
    df_speech = pd.read_pickle(caminho_completo)

    # 2. Ver a estrutura
    print("\n--- Estrutura Geral do Ficheiro de Speech ---")
    df_speech.info()

    # 3. Espreitar as primeiras transcrições
    print("\n--- Primeiras 5 Transcrições do Debate ---")
    # A coluna principal aqui chama-se 'transcript'
    display(df_speech[['transcript']].head())

    # 4. Exemplo de um texto completo
    print("\n--- Exemplo do Segmento 13 (para comparar com o PDF da aula) ---")
    # O PDF usa o índice 13 como exemplo
    print(df_speech.iloc[13]['transcript'])

except FileNotFoundError:
    print(f"Erro: Não encontrei o ficheiro {nome_ficheiro_speech} na pasta {pasta}.")

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

# 1. Criar uma lista de 'stopwords' (palavras comuns que não nos interessam para a análise)
stopwords_pt = ['que', 'de', 'não', 'a', 'o', 'e', 'é', 'um', 'uma', 'os', 'as', 'para',
                'com', 'por', 'do', 'da', 'em', 'no', 'na', 'se', 'mas', 'como', 'mais',
                'ou', 'eu', 'nós', 'ele', 'tem', 'isso', 'este', 'esta', 'são', 'ter', 'ser',
                'foi', 'ao', 'dos', 'das', 'sua', 'seu', 'já', 'essa', 'esse', 'porque',
                'está', 'temos', 'vai', 'seja', 'há', 'quando', 'fazer', 'só', 'acho',
                'faz', 'vão', 'muito', 'coisa', 'estou', 'então']

# 2. Inicializar o contador (Bag of Words)
vectorizer = CountVectorizer(stop_words=stopwords_pt)

# 3. Aplicar aos textos do debate
count_matrix = vectorizer.fit_transform(df_speech['transcript'])

# 4. Obter os nomes das palavras e as contagens totais
palavras = vectorizer.get_feature_names_out()
contagens = count_matrix.toarray().sum(axis=0)

# 5. Juntar tudo num DataFrame para ver o Top
df_palavras = pd.DataFrame({'Palavra': palavras, 'Frequencia': contagens})
df_palavras = df_palavras.sort_values(by='Frequencia', ascending=False)

print("--- Top 50 Palavras Mais Usadas no Debate ---")
display(df_palavras.head(50))

In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Definir os caminhos
pasta = "Project_Features"
ficheiro_polls = "polls_timeline.pkl"
ficheiro_partidos = "candidate_party.pkl"

# 2. Carregar os dados
try:
    df_polls = pd.read_pickle(os.path.join(pasta, ficheiro_polls))
    df_partidos = pd.read_pickle(os.path.join(pasta, ficheiro_partidos))

    # 3. Explorar os Partidos
    print("--- DADOS DOS CANDIDATOS E PARTIDOS ---")
    display(df_partidos.head())

    # 4. Explorar as Sondagens (Polls)
    print("\n--- ESTRUTURA DAS SONDAGENS ---")
    df_polls.info()

    print("\n--- PRIMEIRAS LINHAS DAS SONDAGENS ---")
    display(df_polls.head())
    print("\nNomes das colunas das sondagens:", df_polls.columns.tolist())

except FileNotFoundError as e:
    print(f"Erro ao carregar ficheiros: {e}")
    print("Verifica se os ficheiros 'polls.pkl' e 'candidates_parties.pkl' estão na pasta 'Project_Features'.")

POLLS


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Preparar a tela do gráfico
plt.figure(figsize=(14, 7))
sns.set_theme(style="whitegrid")

# 2. Vamos focar-nos nos 5 candidatos principais
candidatos = ['Gouveia_Melo', 'Marques_Mendes', 'Seguro', 'Ventura', 'Cotrim_Figueiredo']

# 3. Desenhar a evolução de cada um
for candidato in candidatos:
    plt.plot(df_polls['date'], df_polls[candidato], marker='o', linewidth=2.5, label=candidato)

# 4. Formatar o gráfico
plt.title('A Corrida Presidencial: Evolução das Intenções de Voto', fontsize=16, weight='bold')
plt.xlabel('Data da Sondagem', fontsize=12, weight='bold')
plt.ylabel('Intenção de Voto (%)', fontsize=12, weight='bold')


plt.xticks(rotation=45)

plt.legend(title='Candidatos', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import re
import os
import glob

pasta = "Project_Features"
# Encontrar todos os ficheiros de speech
ficheiros_speech = glob.glob(os.path.join(pasta, "*_speech.pkl"))

print(f"A iniciar a varredura global em {len(ficheiros_speech)} ficheiros... A procurar alucinações do Whisper.\n")

resultados_globais = []

# As nossas Expressões Regulares (Regex) para detetar lixo
regex_alien = r'[^\w\s.,!?;:\'"À-ÿ-]'
regex_monstro = r'\b\S{25,}\b'
regex_consoantes = r'[bcdfghjklmnpqrstvwxyzbcç]{5,}'

for caminho in ficheiros_speech:
    nome_ficheiro = os.path.basename(caminho).replace('_speech.pkl', '')

    try:
        df = pd.read_pickle(caminho)

        # Ignorar ficheiros vazios para não dar erro
        if len(df) == 0:
            continue

        # Garantir que tudo é texto e não há nulos
        df['transcript_clean'] = df['transcript'].astype(str).fillna('')

        # Aplicar os 3 filtros
        df['tem_alien'] = df['transcript_clean'].apply(lambda x: bool(re.search(regex_alien, x)))
        df['tem_monstro'] = df['transcript_clean'].apply(lambda x: bool(re.search(regex_monstro, x)))
        df['tem_consoantes'] = df['transcript_clean'].apply(lambda x: bool(re.search(regex_consoantes, x.lower())))

        # Contabilizar os estragos
        total_aliens = df['tem_alien'].sum()
        total_monstros = df['tem_monstro'].sum()
        total_consoantes = df['tem_consoantes'].sum()

        # Quantas linhas (segmentos) têm pelo menos um destes erros?
        linhas_corrompidas = len(df[(df['tem_alien']) | (df['tem_monstro']) | (df['tem_consoantes'])])

        resultados_globais.append({
            'Ficheiro': nome_ficheiro,
            'Total Segmentos': len(df),
            'Linhas com Lixo': linhas_corrompidas,
            'Carateres Estranhos': total_aliens,
            'Palavras Monstro (>25 letras)': total_monstros,
            'Loops Consoantes': total_consoantes
        })

    except Exception as e:
        print(f"Erro no ficheiro {nome_ficheiro}: {e}")

# Criar a tabela final com o resumo de todos os ficheiros
df_resumo = pd.DataFrame(resultados_globais)

print("=====================================================")
print("     AUDITORIA FINAL: ALUCINAÇÕES E LIXO TEXTUAL     ")
print("=====================================================\n")

print(f"Total de ficheiros analisados: {len(df_resumo)}")
print(f"Total de linhas (segmentos) de fala comprometidos em todo o dataset: {df_resumo['Linhas com Lixo'].sum()}\n")

print("🚨 TOP 10 FICHEIROS COM MAIS LIXO NO TEXTO:")
# Mostrar os piores ofensores, ordenados por número de linhas corrompidas
display(df_resumo.sort_values(by='Linhas com Lixo', ascending=False).head(42))

In [ ]:
import os
import pandas as pd

pasta_features = 'Project_Features'
pklfiles = [f for f in os.listdir(pasta_features) if os.path.isfile(os.path.join(pasta_features, f)) and f.endswith('_speech.pkl')]
print(f"Encontrados {len(pklfiles)} ficheiros speech.")

# Lista temporária para guardar os DataFrames individuais
lista_dfs = []

for ficheiro in pklfiles:
    caminho_completo = os.path.join(pasta_features, ficheiro)

    # 1. Carregar o ficheiro pkl
    df_temp = pd.read_pickle(caminho_completo)

    # 2. DICA DE OURO: Criar uma coluna para identificar a origem!
    # Quando juntares todos os ficheiros, vais precisar de saber a que vídeo/telejornal
    # pertence cada segmento de fala.
    nome_video = ficheiro.replace('_speech.pkl', '')
    df_temp['video_id'] = nome_video

    # 3. Adicionar o DataFrame à lista
    lista_dfs.append(df_temp)

# 4. Juntar todos os DataFrames num único grande DataFrame (O teu dataset final!)
if lista_dfs:
    df_principal = pd.concat(lista_dfs, ignore_index=True)
    print(f"\nDataFrame principal criado com sucesso! Tem {len(df_principal)} linhas e {len(df_principal.columns)} colunas.")

    # Ver as primeiras 5 linhas para confirmar que está tudo bem
    display(df_principal.head())
else:
    print("Aviso: Nenhum ficheiro foi carregado. Verifica o caminho da pasta.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

print("A calcular novas métricas de fala...")

# 0. CRIAR A COLUNA word_count PRIMEIRO!
df_principal['word_count'] = df_principal['transcript'].apply(lambda x: len(str(x).split()))

# 1. Calcular o Speech Rate (Palavras por Segundo)
# Evitamos a divisão por zero caso haja durações de 0.0 segundos
df_principal['speech_rate'] = df_principal['word_count'] / df_principal['duration'].replace(0, np.nan)

# 2. Calcular a Densidade Lexical
# Função simples para contar palavras únicas num texto
def calcular_densidade_lexical(texto):
    if pd.isna(texto):
        return 0
    palavras = str(texto).lower().split()
    if len(palavras) == 0:
        return 0
    palavras_unicas = set(palavras)
    return len(palavras_unicas) / len(palavras)

df_principal['lexical_density'] = df_principal['transcript'].apply(calcular_densidade_lexical)

print("Métricas calculadas com sucesso!")

# 3. Vamos visualizar! Quem fala mais rápido nos diferentes vídeos?
plt.figure(figsize=(14, 7))
sns.set_theme(style="whitegrid")

# Usamos um Boxplot para ver a distribuição da velocidade de fala por vídeo
# Filtramos outliers extremos (ex: taxa de fala irrealista > 10) para o gráfico ficar legível
df_filtrado = df_principal[df_principal['speech_rate'] < 10]

ax = sns.boxplot(
    data=df_filtrado,
    x='speech_rate',
    y='video_id',
    palette='viridis'
)

plt.title("Distribuição da Taxa de Fala (Palavras por Segundo) por Vídeo", fontsize=16, pad=15)
plt.xlabel("Palavras por Segundo (Speech Rate)", fontsize=12)
plt.ylabel("Vídeo / Debate", fontsize=12)
plt.tight_layout()

plt.show()

# Ver um resumo estatístico das novas métricas
print("\n--- Resumo Estatístico das Novas Métricas ---")
display(df_principal[['speech_rate', 'lexical_density']].describe())

In [ ]:
# 1. Criar uma função ou regra para classificar o tipo de vídeo
# ATENÇÃO: Ajusta esta regra consoante o nome real dos teus ficheiros!
def classificar_tipo_video(video_id):
    video_id_lower = str(video_id).lower()
    # Se o nome do ficheiro tiver 'vs', assumimos que é um debate
    if 'vs' in video_id_lower or 'debate' in video_id_lower:
        return 'Debate'
    else:
        # Caso contrário, assumimos que é telejornal
        return 'Telejornal'

# Criar a nova coluna macro
df_principal['tipo_video'] = df_principal['video_id'].apply(classificar_tipo_video)

# 2. Separar em dois DataFrames distintos
df_debates = df_principal[df_principal['tipo_video'] == 'Debate'].copy()
df_telejornais = df_principal[df_principal['tipo_video'] == 'Telejornal'].copy()

print(f"Temos {len(df_debates)} segmentos de Debates e {len(df_telejornais)} segmentos de Telejornais.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.set_theme(style="whitegrid")

# Vamos comparar a Taxa de Fala (Palavras por Segundo) entre as duas categorias
sns.boxplot(
    data=df_principal[df_principal['speech_rate'] < 10],
    x='tipo_video',
    y='speech_rate',
    palette='Set2'
)

plt.title("Comparação da Taxa de Fala: Debates vs Telejornais", fontsize=16)
plt.xlabel("Categoria do Vídeo", fontsize=12)
plt.ylabel("Palavras por Segundo (Speech Rate)", fontsize=12)
plt.show()

In [ ]:
import os
import pandas as pd
import numpy as np

pasta_features = 'Project_Features'
pklfiles = [f for f in os.listdir(pasta_features) if os.path.isfile(os.path.join(pasta_features, f)) and f.endswith('_speech.pkl')]

# 1. Criar listas separadas para guardar os dados
lista_debates = []
lista_telejornais = []

print("A processar e a separar os ficheiros...")

for ficheiro in pklfiles:
    caminho_completo = os.path.join(pasta_features, ficheiro)
    df_temp = pd.read_pickle(caminho_completo)

    # Extrair o nome do vídeo
    nome_video = ficheiro.replace('_speech.pkl', '')
    df_temp['video_id'] = nome_video

    # 2. Regra de Separação (Ajusta 'vs' se os teus ficheiros tiverem outro padrão)
    if 'vs' in nome_video.lower() or 'debate' in nome_video.lower():
        lista_debates.append(df_temp)
    else:
        lista_telejornais.append(df_temp)

# 3. Criar os dois DataFrames independentes
df_debates = pd.concat(lista_debates, ignore_index=True) if lista_debates else pd.DataFrame()
df_telejornais = pd.concat(lista_telejornais, ignore_index=True) if lista_telejornais else pd.DataFrame()

# 4. Função para calcular as nossas métricas avançadas de forma limpa
def engenharia_de_features(df):
    if df.empty:
        return df

    # Remover linhas sem transcrição
    df = df.dropna(subset=['transcript']).copy()

    # Calcular Número de Palavras
    df['word_count'] = df['transcript'].apply(lambda x: len(str(x).split()))

    # Calcular Taxa de Fala (Palavras por Segundo)
    df['speech_rate'] = df['word_count'] / df['duration'].replace(0, np.nan)

    # Calcular Densidade Lexical
    def densidade(texto):
        palavras = str(texto).lower().split()
        if len(palavras) == 0: return 0
        return len(set(palavras)) / len(palavras)

    df['lexical_density'] = df['transcript'].apply(densidade)

    return df

# Aplicar a função a ambos os DataFrames
df_debates = engenharia_de_features(df_debates)
df_telejornais = engenharia_de_features(df_telejornais)

print("\n=== SUCESSO ===")
print(f"Debates carregados: {len(df_debates)} segmentos de fala.")
print(f"Telejornais carregados: {len(df_telejornais)} segmentos de fala.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer
import nltk
from nltk.corpus import stopwords

# 1. Descarregar e preparar a lista de Stopwords em Português
nltk.download('stopwords', quiet=True)
stop_words_pt = stopwords.words('portuguese')

# DICA PRO: Em linguagem falada há sempre palavras de "muleta" que escapam.
# Vamos adicionar algumas à nossa lista de bloqueio para limpar o gráfico:
palavras_extra_bloqueadas = ['que', 'de', 'não', 'a', 'o', 'e', 'é', 'um', 'uma', 'os', 'as', 'para',
                'com', 'por', 'do', 'da', 'em', 'no', 'na', 'se', 'mas', 'como', 'mais',
                'ou', 'eu', 'nós', 'ele', 'tem', 'isso', 'este', 'esta', 'são', 'ter', 'ser',
                'foi', 'ao', 'dos', 'das', 'sua', 'seu', 'já', 'essa', 'esse', 'porque',
                'está', 'temos', 'vai', 'seja', 'há', 'quando', 'fazer', 'só', 'acho',
                'faz', 'vão', 'muito', 'coisa', 'estou', 'então', 'vou', 'dizer', 'aqui',
                'agora', 'ainda', 'sobre', 'pode', 'bem', 'portanto', 'pode', 'assim', 'todos',
                'quero', 'quer', 'dia', 'todos', 'bem', 'mal', 'têm', 'sei', 'ver', 'anos',
                'dois', 'disse', 'dar', 'vamos', 'neste', 'primeiro', 'onde', 'tudo']
stop_words_pt.extend(palavras_extra_bloqueadas)

# 2. Função rápida para contar o Top N palavras de um DataFrame
def extrair_top_palavras(textos, top_n=15):
    # O CountVectorizer converte texto numa matriz de contagem
    vetorizador = CountVectorizer(stop_words=stop_words_pt, max_df=0.9)
    matriz_palavras = vetorizador.fit_transform(textos)

    # Somar as ocorrências de cada palavra
    soma_palavras = matriz_palavras.sum(axis=0)

    # Criar uma lista com (palavra, frequência)
    frequencias = [(palavra, soma_palavras[0, idx]) for palavra, idx in vetorizador.vocabulary_.items()]

    # Ordenar do maior para o menor e converter num DataFrame pequeno
    frequencias = sorted(frequencias, key=lambda x: x[1], reverse=True)
    return pd.DataFrame(frequencias[:top_n], columns=['Palavra', 'Frequencia'])

# 3. Aplicar a função às transcrições dos dois DataFrames
# Asseguramos que convertemos tudo para string e removemos eventuais nulos
top_debates = extrair_top_palavras(df_debates['transcript'].dropna().astype(str))
top_telejornais = extrair_top_palavras(df_telejornais['transcript'].dropna().astype(str))

# 4. Desenhar os Gráficos Lado a Lado
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
sns.set_theme(style="whitegrid")

# Gráfico da Esquerda: Debates
sns.barplot(data=top_debates, x='Frequencia', y='Palavra', ax=axes[0], palette='Reds_r')
axes[0].set_title('Top 15 Palavras - DEBATES', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Número de Ocorrências')
axes[0].set_ylabel('')

# Gráfico da Direita: Telejornais
sns.barplot(data=top_telejornais, x='Frequencia', y='Palavra', ax=axes[1], palette='Blues_r')
axes[1].set_title('Top 15 Palavras - TELEJORNAIS', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Número de Ocorrências')
axes[1].set_ylabel('')

# Ajustar o layout para não sobrepor letras
plt.tight_layout()
plt.show()

In [ ]:
print("--- Top 15 Telejornais ---")
print(top_telejornais.head(15))

print("\n--- Top 15 Debates ---")
print(top_debates.head(15))

In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("A calcular o Índice de Personalização...")

# Função para contar pronomes da 1ª pessoa e dividir pelo total de palavras
def calcular_personalizacao(texto):
    if pd.isna(texto):
        return 0.0
    texto = str(texto).lower()

    # Procura palavras exatas: eu, me, mim, comigo, meu, minha, meus, minhas
    pronomes_1a = len(re.findall(r'\b(eu|me|mim|comigo|meu|minha|meus|minhas)\b', texto))

    palavras = len(texto.split())
    if palavras == 0:
        return 0.0

    # Retorna a percentagem de palavras que são pronomes da 1ª pessoa
    return (pronomes_1a / palavras) * 100

# Aplicar aos dois DataFrames
df_debates['personalization_ratio'] = df_debates['transcript'].apply(calcular_personalizacao)
df_telejornais['personalization_ratio'] = df_telejornais['transcript'].apply(calcular_personalizacao)

print("Índice de Personalização calculado!")

In [ ]:
print("A calcular a Dinâmica de Interrupções...")

def calcular_interrupcoes(df):
    if df.empty: return df

    # 1. Ordenar por vídeo e depois cronologicamente pelo tempo inicial
    df = df.sort_values(by=['video_id', 'timestamp']).copy()

    # 2. Calcular o tempo exato em que o segmento acaba
    df['end_time'] = df['timestamp'] + df['duration']

    # 3. Descobrir o tempo em que o segmento *anterior* acabou (agrupado por vídeo)
    df['prev_end_time'] = df.groupby('video_id')['end_time'].shift(1)

    # 4. Calcular o Time Gap (Tempo do início atual - Tempo do fim anterior)
    # Se o gap for negativo, a pessoa começou a falar antes da outra se calar!
    df['time_gap'] = df['timestamp'] - df['prev_end_time']

    # 5. Criar a flag de Interrupção (Gap negativo)
    # Ignoramos gaps perfeitamente nulos ou gigantes para limpar possíveis erros de anotação
    df['is_interruption'] = (df['time_gap'] < 0) & (df['time_gap'] > -10)

    return df

# Aplicar o cálculo temporal
df_debates = calcular_interrupcoes(df_debates)
df_telejornais = calcular_interrupcoes(df_telejornais)

print("Métricas de Interrupção calculadas!")

In [ ]:
# Criar uma figura com 2 gráficos lado a lado
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.set_theme(style="whitegrid")

# Gráfico 1: Comparar o Índice de Personalização
sns.barplot(
    data=pd.DataFrame({
        'Formato': ['Debates', 'Telejornais'],
        'Personalização (%)': [df_debates['personalization_ratio'].mean(), df_telejornais['personalization_ratio'].mean()]
    }),
    x='Formato', y='Personalização (%)', ax=axes[0], palette='muted'
)
axes[0].set_title('Uso de Pronomes na 1ª Pessoa ("Eu", "Meu")', fontsize=14)
axes[0].set_ylabel('% do Total de Palavras')

# Gráfico 2: Contagem de Interrupções
interrupcoes_debates = df_debates['is_interruption'].sum()
interrupcoes_telejornais = df_telejornais['is_interruption'].sum()

sns.barplot(
    x=['Debates', 'Telejornais'],
    y=[interrupcoes_debates, interrupcoes_telejornais],
    ax=axes[1], palette='flare'
)
axes[1].set_title('Número Total de Interrupções Detetadas', fontsize=14)
axes[1].set_ylabel('Quantidade Absoluta')

# Adicionar os números em cima das barras para o gráfico de interrupções
for container in axes[1].containers:
    axes[1].bar_label(container, padding=3)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("A calcular o Ranking de Dinâmica dos Debates...")

# 1. Agrupar os dados por Debate (video_id)
# Vamos calcular o Total de Interrupções e a Média do Índice de Personalização
ranking_debates = df_debates.groupby('video_id').agg(
    total_interrupcoes=('is_interruption', 'sum'),
    media_personalizacao=('personalization_ratio', 'mean'),
    media_speech_rate=('speech_rate', 'mean')
).reset_index()

# 2. Ordenar o ranking do debate MAIS interrompido para o MENOS interrompido
ranking_debates = ranking_debates.sort_values(by='total_interrupcoes', ascending=False)

# 3. Vamos visualizar o Top 10 dos Debates mais "Tensos"
top_28_tensos = ranking_debates.head(28)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
sns.set_theme(style="whitegrid")

# Gráfico da Esquerda: Total de Interrupções
sns.barplot(
    data=top_28_tensos,
    y='video_id',
    x='total_interrupcoes',
    ax=axes[0],
    palette='Reds_r'
)
axes[0].set_title('Top 28 Debates com Mais Interrupções', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Número Total de Interrupções')
axes[0].set_ylabel('Debates (video_id)')

# Adicionar os números nas barras
for container in axes[0].containers:
    axes[0].bar_label(container, padding=3)

# Gráfico da Direita: Índice de Personalização dos mesmos debates
sns.barplot(
    data=top_28_tensos,
    y='video_id',
    x='media_personalizacao',
    ax=axes[1],
    palette='Blues_r'
)
axes[1].set_title('Média de Uso do "Eu" nestes mesmos Debates', fontsize=14, fontweight='bold')
axes[1].set_xlabel('% Média de Pronomes Pessoais')
axes[1].set_ylabel('')

# Ocultar os nomes do eixo Y no segundo gráfico para ficar mais limpo
axes[1].set_yticklabels([])

plt.tight_layout()
plt.show()

In [ ]:
print(top_28_tensos)

In [ ]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import seaborn as sns

print("A construir o perfil de cada candidato...")

registos_candidatos = []

# Iterar sobre o dataframe que tinhas antes (assumindo que se chama 'ranking_debates')
for index, row in ranking_debates.iterrows():
    video = row['video_id']

    # Usar Expressões Regulares (Regex) para extrair os dois nomes antes do mês
    # Exemplo: 'Seguro_vs_Martins_December_6' -> Cand1: 'Seguro', Cand2: 'Martins'
    match = re.search(r'(.+?)_vs_(.+?)_(?:January|February|March|April|May|June|July|August|September|October|November|December)', video)

    if match:
        # Extrair e limpar os nomes (trocar _ por espaço para ficar bonito nos gráficos)
        cand1 = match.group(1).replace('_', ' ')
        cand2 = match.group(2).replace('_', ' ')
    else:
        continue

    # Aplicar a tua regra de ouro: metade da responsabilidade nas interrupções!
    metade_interrupcoes = row['total_interrupcoes'] / 2.0
    pers = row['media_personalizacao']
    speech = row['media_speech_rate']

    # Guardar os valores para o Candidato 1
    registos_candidatos.append({'Candidato': cand1, 'interrupcoes_atribuidas': metade_interrupcoes, 'personalizacao': pers, 'speech_rate': speech})
    # Guardar os mesmos valores para o Candidato 2
    registos_candidatos.append({'Candidato': cand2, 'interrupcoes_atribuidas': metade_interrupcoes, 'personalizacao': pers, 'speech_rate': speech})

# Criar um DataFrame temporário com todos os registos separados
df_candidatos_raw = pd.DataFrame(registos_candidatos)

# Agrupar agora por Candidato para tirar as médias finais da campanha de cada um!
df_perfil_candidato = df_candidatos_raw.groupby('Candidato').agg(
    total_interrupcoes=('interrupcoes_atribuidas', 'sum'),
    media_interrupcoes_por_debate=('interrupcoes_atribuidas', 'mean'),
    media_personalizacao=('personalizacao', 'mean'),
    media_speech_rate=('speech_rate', 'mean'),
    num_debates=('Candidato', 'count') # Contar a quantos debates foi
).reset_index()

# Ordenar para ver quem é o mais "interruptor" em média
df_perfil_candidato = df_perfil_candidato.sort_values(by='media_interrupcoes_por_debate', ascending=False)

print("\n--- Perfil Médio de Cada Candidato ---")
display(df_perfil_candidato.round(2))

In [ ]:
plt.figure(figsize=(12, 8))
sns.set_theme(style="whitegrid")

# Criar o Scatter Plot
sns.scatterplot(
    data=df_perfil_candidato,
    x='media_personalizacao',
    y='media_interrupcoes_por_debate',
    s=300, # Tamanho da "bolha"
    color='royalblue',
    alpha=0.8
)

# Adicionar os nomes dos candidatos colados às bolhas
for i in range(df_perfil_candidato.shape[0]):
    plt.text(
        x=df_perfil_candidato['media_personalizacao'].iloc[i] + 0.05,
        y=df_perfil_candidato['media_interrupcoes_por_debate'].iloc[i],
        s=df_perfil_candidato['Candidato'].iloc[i],
        fontdict=dict(size=11, weight='bold')
    )

# Linhas médias para criar os "Quadrantes"
plt.axvline(x=df_perfil_candidato['media_personalizacao'].mean(), color='red', linestyle='--', alpha=0.5)
plt.axhline(y=df_perfil_candidato['media_interrupcoes_por_debate'].mean(), color='red', linestyle='--', alpha=0.5)

plt.title('Matriz de Comportamento dos Candidatos: Agressividade vs Foco Pessoal', fontsize=16, pad=20)
plt.xlabel('Foco Pessoal (% de pronomes da 1ª pessoa)', fontsize=12)
plt.ylabel('Média de Interrupções Atribuídas por Debate', fontsize=12)

# Adicionar legendas aos quadrantes (Opcional, mas dá muito estilo!)
plt.text(plt.xlim()[1]*0.8, plt.ylim()[1]*0.95, 'Mais Agressivos & Egocêntricos', color='grey', style='italic')

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

print("A extrair os 5 principais temas/palavras-chave por debate...")

# 1. Agrupar o texto de cada debate (como fizemos antes)
textos_por_debate = df_debates.groupby('video_id')['transcript'].apply(
    lambda x: ' '.join(x.dropna().astype(str))
).reset_index()

# 2. Configurar o TF-IDF
# Usamos o TF-IDF porque ele ignora as palavras genéricas e foca-se no que é ÚNICO daquele debate
vetorizador = TfidfVectorizer(stop_words=stop_words_pt)
matriz_tfidf = vetorizador.fit_transform(textos_por_debate['transcript'])
nomes_palavras = vetorizador.get_feature_names_out()

# 3. Descobrir o Top 5 de cada debate
lista_resultados = []

# Iterar sobre cada debate
for i, linha in textos_por_debate.iterrows():
    nome_video = linha['video_id']

    # Obter as pontuações TF-IDF específicas deste debate (linha i da matriz)
    pontuacoes = matriz_tfidf[i].toarray()[0]

    # Juntar cada palavra com a sua pontuação e ordenar da maior para a menor
    palavras_com_pontuacao = list(zip(nomes_palavras, pontuacoes))
    palavras_com_pontuacao.sort(key=lambda x: x[1], reverse=True)

    # Extrair apenas as 5 palavras (temas) mais fortes
    top_5_palavras = [palavra for palavra, pontuacao in palavras_com_pontuacao[:5]]

    # Limpar o nome do vídeo para a tabela ficar bonita (tirar underscores e a data)
    # Ex: 'Seguro_vs_Martins_December_6' -> 'Seguro vs Martins'
    import re
    match = re.search(r'(.+?)_vs_(.+?)_', nome_video)
    nome_limpo = f"{match.group(1)} vs {match.group(2)}".replace('_', ' ') if match else nome_video

    # Guardar na nossa lista final
    lista_resultados.append([nome_limpo] + top_5_palavras)

# 4. Criar e mostrar a Tabela Final
df_top_temas = pd.DataFrame(
    lista_resultados,
    columns=['Frente-a-Frente', 'Tema 1', 'Tema 2', 'Tema 3', 'Tema 4', 'Tema 5']
)

print("\n=== A IDENTIDADE DE CADA DEBATE ===")
display(df_top_temas)

In [ ]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import seaborn as sns

print("A aplicar o Dicionário Central de Temas aos Debates...")

# 1. Colar aqui o dicionário do teu colega (apenas os temas para esta análise)
THEME_ALIASES = {
    "Eleições/Campanha": ["presidenciais", "presidencial", "eleições", "eleicao", "eleições presidenciais", "eleicoes presidenciais", "candidato", "candidata", "candidatos", "candidatas", "candidatura", "candidaturas", "campanha", "abstenção", "abstencao", "urna", "urnas", "eleitor", "eleitores", "debate", "debates"],
    "Sondagens": ["sondagem", "sondagens", "barómetro", "barometro", "voto", "votos", "intenção de voto", "intencao de voto", "intenções de voto", "intencoes de voto", "projecção", "projeccao", "projeção", "projecao"],
    "Governo/Partidos": ["governo", "primeiro ministro", "primeira ministra", "montenegro", "luís montenegro", "luis montenegro", "parlamento", "assembleia", "partido", "partidos", "oposição", "oposicao"],
    "Saúde": ["saúde", "saude", "sns", "hospital", "hospitais", "médico", "medico", "médicos", "medicos", "enfermeiro", "enfermeiros", "urgência", "urgencias", "inem", "listas de espera", "lista de espera", "ambulância", "ambulancia"],
    "Economia": ["economia", "inflação", "inflacao", "preços", "precos", "imposto", "impostos", "irs", "orçamento", "orcamento", "salário", "salario", "salários", "salarios", "rendimento", "pib", "juros", "bce"],
    "Habitação": ["habitação", "habitacao", "casa", "casas", "arrendamento", "renda", "rendas", "senhorio", "senhorios", "inquilino", "inquilinos", "crédito habitação", "credito habitacao"],
    "Educação": ["educação", "educacao", "escola", "escolas", "professor", "professores", "aluno", "alunos", "ensino", "aulas", "creche", "creches"],
    "Justiça/Segurança": ["justiça", "justica", "tribunal", "tribunais", "polícia", "policia", "pj", "psp", "gnr", "segurança", "seguranca", "crime", "crimes", "corrupção", "corrupcao", "pgr", "ministério público", "ministerio publico"],
    "Internacional": ["ucrânia", "ucrania", "rússia", "russia", "guerra", "israel", "gaza", "palestina", "trump", "eua", "estados unidos", "europa", "bruxelas", "união europeia", "uniao europeia"],
    "Greves/Trabalho": ["greve", "greves", "sindicato", "sindicatos", "trabalhador", "trabalhadores", "protesto", "protestos", "manifestação", "manifestacao", "manifestantes", "patrões", "patroes"]
}

# 2. Função para contar quantas vezes um TEMA é mencionado num texto
def contar_temas(texto):
    if pd.isna(texto):
        return {tema: 0 for tema in THEME_ALIASES.keys()}

    texto = str(texto).lower()
    contagens = {}

    for tema, aliases in THEME_ALIASES.items():
        total_mencoes = 0
        for alias in aliases:
            # Usamos \b para garantir que apanhamos a palavra exata (não apanha "casaco" se procurarmos "casa")
            padrao = r'\b' + re.escape(alias.lower()) + r'\b'
            total_mencoes += len(re.findall(padrao, texto))
        contagens[tema] = total_mencoes

    return contagens

# 3. Vamos juntar todas as falas de cada debate num só texto para analisar o peso dos temas
textos_por_debate = df_debates.groupby('video_id')['transcript'].apply(lambda x: ' '.join(x.dropna().astype(str))).reset_index()

# 4. Aplicar a nossa função de contagem
textos_por_debate['contagem_temas'] = textos_por_debate['transcript'].apply(contar_temas)

# 5. Transformar os dicionários de resultados em colunas para podermos somar e desenhar gráficos
df_temas = pd.json_normalize(textos_por_debate['contagem_temas'])
df_temas['video_id'] = textos_por_debate['video_id']

# 6. O GRÁFICO FINAL: Quais foram os temas mais discutidos em todos os debates?
# Somar todas as colunas de temas
temas_totais = df_temas.drop(columns=['video_id']).sum().sort_values(ascending=False).reset_index()
temas_totais.columns = ['Tema Oficial', 'Nº Total de Menções']

plt.figure(figsize=(12, 6))
sns.set_theme(style="whitegrid")

sns.barplot(
    data=temas_totais,
    y='Tema Oficial',
    x='Nº Total de Menções',
    palette='viridis'
)

plt.title('Temas Dominantes na Campanha (Baseado no Dicionário Uniformizado)', fontsize=16, fontweight='bold')
plt.xlabel('Número Total de Vezes que o Tema foi Falado', fontsize=12)
plt.ylabel('')

# Adicionar os números nas barras
for i, v in enumerate(temas_totais['Nº Total de Menções']):
    plt.text(v + 5, i, str(v), color='black', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import seaborn as sns

print("A aplicar o Dicionário Central de Temas aos telejornais...")

# 1. Colar aqui o dicionário do teu colega (apenas os temas para esta análise)
THEME_ALIASES = {
    "Eleições/Campanha": ["presidenciais", "presidencial", "eleições", "eleicao", "eleições presidenciais", "eleicoes presidenciais", "candidato", "candidata", "candidatos", "candidatas", "candidatura", "candidaturas", "campanha", "abstenção", "abstencao", "urna", "urnas", "eleitor", "eleitores", "debate", "debates"],
    "Sondagens": ["sondagem", "sondagens", "barómetro", "barometro", "voto", "votos", "intenção de voto", "intencao de voto", "intenções de voto", "intencoes de voto", "projecção", "projeccao", "projeção", "projecao"],
    "Governo/Partidos": ["governo", "primeiro ministro", "primeira ministra", "montenegro", "luís montenegro", "luis montenegro", "parlamento", "assembleia", "partido", "partidos", "oposição", "oposicao"],
    "Saúde": ["saúde", "saude", "sns", "hospital", "hospitais", "médico", "medico", "médicos", "medicos", "enfermeiro", "enfermeiros", "urgência", "urgencias", "inem", "listas de espera", "lista de espera", "ambulância", "ambulancia"],
    "Economia": ["economia", "inflação", "inflacao", "preços", "precos", "imposto", "impostos", "irs", "orçamento", "orcamento", "salário", "salario", "salários", "salarios", "rendimento", "pib", "juros", "bce"],
    "Habitação": ["habitação", "habitacao", "casa", "casas", "arrendamento", "renda", "rendas", "senhorio", "senhorios", "inquilino", "inquilinos", "crédito habitação", "credito habitacao"],
    "Educação": ["educação", "educacao", "escola", "escolas", "professor", "professores", "aluno", "alunos", "ensino", "aulas", "creche", "creches"],
    "Justiça/Segurança": ["justiça", "justica", "tribunal", "tribunais", "polícia", "policia", "pj", "psp", "gnr", "segurança", "seguranca", "crime", "crimes", "corrupção", "corrupcao", "pgr", "ministério público", "ministerio publico"],
    "Internacional": ["ucrânia", "ucrania", "rússia", "russia", "guerra", "israel", "gaza", "palestina", "trump", "eua", "estados unidos", "europa", "bruxelas", "união europeia", "uniao europeia"],
    "Greves/Trabalho": ["greve", "greves", "sindicato", "sindicatos", "trabalhador", "trabalhadores", "protesto", "protestos", "manifestação", "manifestacao", "manifestantes", "patrões", "patroes"]
}

# 2. Função para contar quantas vezes um TEMA é mencionado num texto
def contar_temas(texto):
    if pd.isna(texto):
        return {tema: 0 for tema in THEME_ALIASES.keys()}

    texto = str(texto).lower()
    contagens = {}

    for tema, aliases in THEME_ALIASES.items():
        total_mencoes = 0
        for alias in aliases:
            # Usamos \b para garantir que apanhamos a palavra exata (não apanha "casaco" se procurarmos "casa")
            padrao = r'\b' + re.escape(alias.lower()) + r'\b'
            total_mencoes += len(re.findall(padrao, texto))
        contagens[tema] = total_mencoes

    return contagens

# 3. Vamos juntar todas as falas de cada debate num só texto para analisar o peso dos temas
textos_por_telejornal = df_telejornais.groupby('video_id')['transcript'].apply(lambda x: ' '.join(x.dropna().astype(str))).reset_index()

# 4. Aplicar a nossa função de contagem
textos_por_telejornal['contagem_temas'] = textos_por_telejornal['transcript'].apply(contar_temas)

# 5. Transformar os dicionários de resultados em colunas para podermos somar e desenhar gráficos
df_temas = pd.json_normalize(textos_por_telejornal['contagem_temas'])
df_temas['video_id'] = textos_por_telejornal['video_id']

# 6. O GRÁFICO FINAL: Quais foram os temas mais discutidos em todos os debates?
# Somar todas as colunas de temas
temas_totais = df_temas.drop(columns=['video_id']).sum().sort_values(ascending=False).reset_index()
temas_totais.columns = ['Tema Oficial', 'Nº Total de Menções']

plt.figure(figsize=(12, 6))
sns.set_theme(style="whitegrid")

sns.barplot(
    data=temas_totais,
    y='Tema Oficial',
    x='Nº Total de Menções',
    palette='viridis'
)

plt.title('Temas Dominantes na Campanha (Baseado no Dicionário Uniformizado)', fontsize=16, fontweight='bold')
plt.xlabel('Número Total de Vezes que o Tema foi Falado', fontsize=12)
plt.ylabel('')

# Adicionar os números nas barras
for i, v in enumerate(temas_totais['Nº Total de Menções']):
    plt.text(v + 5, i, str(v), color='black', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Função para converter o nome do vídeo numa data real
def converter_para_data(video_id):
    meses_map = {
        "November": 11, "Nov": 11,
        "December": 12, "Dec": 12,
        "January": 1, "Jan": 1
    }
    # Procura o mês e o dia no fim da string (ex: December_19)
    match = re.search(r'([A-Za-z]+)_(\d+)$', video_id)
    if match:
        mes_nome = match.group(1)
        dia = int(match.group(2))
        mes = meses_map.get(mes_nome, 12)
        ano = 2026 if mes == 1 else 2025 # Ajuste para datas de Janeiro
        return pd.Timestamp(year=ano, month=mes, day=dia)
    return pd.NaT

# 2. Preparar o DataFrame de evolução
# Usamos o df_temas que criámos no passo anterior (que tem as contagens e o video_id)
df_evolucao = df_temas.copy()
df_evolucao['data'] = df_evolucao['video_id'].apply(converter_para_data)

# 3. Agregar por data (somar menções de todos os debates do mesmo dia)
df_timeline = df_evolucao.groupby('data').sum(numeric_only=True).sort_index()

# 4. Desenhar o Gráfico de Linhas
plt.figure(figsize=(15, 7))
sns.set_theme(style="whitegrid")

# Vamos desenhar apenas os temas mais relevantes para não poluir o gráfico
temas_interesse = ["Economia", "Saúde", "Justiça/Segurança", "Eleições/Campanha", "Governo/Partidos", "Greves/Trabalho", "Internacional", "Habitação", "Sondagens", "Educação"]

for tema in temas_interesse:
    if tema in df_timeline.columns:
        sns.lineplot(data=df_timeline, x=df_timeline.index, y=tema, label=tema, marker='o', linewidth=2)

plt.title('Evolução da Agenda Política: Frequência de Temas ao Longo do Tempo', fontsize=16, fontweight='bold')
plt.xlabel('Data do Debate', fontsize=12)
plt.ylabel('Número de Menções (Speech)', fontsize=12)
plt.legend(title="Temas Principais", bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

print("A desenhar o Mapa de Calor Temporal...")

plt.figure(figsize=(14, 8))

# O Heatmap precisa dos temas nas linhas e das datas nas colunas, por isso fazemos a Transposta (.T)
# Formatar as datas no índice das colunas para ficar legível (Ex: '06 Dez')
df_heatmap = df_timeline.T
df_heatmap.columns = df_heatmap.columns.strftime('%d %b')

# Criar o mapa
sns.heatmap(
    df_heatmap,
    cmap='Reds',          # Branco/Laranja/Vermelho
    annot=True,           # Mostra o número exato dentro de cada quadrado
    fmt='g',              # Formato de número inteiro
    linewidths=.5,
    cbar_kws={'label': 'Número de Menções'}
)

plt.title('Mapa de Calor da Agenda Política: Intensidade Temática Diária', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Data do Telejornal', fontsize=12)
plt.ylabel('Temas', fontsize=12)

# Rodar os labels das datas para se lerem bem
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import re

print("A desenhar o Raio-X Temático por Debate...")

# 1. Limpar os nomes dos debates para ficarem legíveis no eixo Y
def limpar_nome(video_id):
    match = re.search(r'(.+?)_vs_(.+?)_', video_id)
    if match:
        return f"{match.group(1).replace('_', ' ')} vs {match.group(2).replace('_', ' ')}"
    return video_id

# Fazer uma cópia limpa dos dados apenas com os temas
df_grafico = df_temas.copy()
df_grafico = df_grafico.drop(columns=['data', 'contagem_temas'], errors='ignore')
df_grafico['Frente-a-Frente'] = df_grafico['video_id'].apply(limpar_nome)
df_grafico = df_grafico.set_index('Frente-a-Frente').drop(columns=['video_id'])

# 2. A Magia: Converter contagens absolutas em Percentagens (0 a 100%) por linha
df_percentagens = df_grafico.div(df_grafico.sum(axis=1), axis=0) * 100

# Ordenar o gráfico com base no tema mais falado no geral (ex: Economia)
# Isto cria um efeito visual muito mais organizado, do debate que menos falou disso para o que mais falou
tema_dominante = df_percentagens.mean().idxmax()
df_percentagens = df_percentagens.sort_values(by=tema_dominante, ascending=True)

# 3. Desenhar o Gráfico de Barras Empilhadas
ax = df_percentagens.plot(
    kind='barh',
    stacked=True,
    figsize=(14, 12),
    cmap='tab10',  # Uma paleta de cores forte e distinta
    width=0.85
)

# 4. Formatação Profissional
plt.title('Temas nos Debates', fontsize=18, fontweight='bold', pad=20)
plt.xlabel('Proporção do Telejornal(%)', fontsize=12)
plt.ylabel('Debates', fontsize=12)

# Colocar a legenda fora do gráfico para não tapar os dados
plt.legend(title='Temas Oficiais', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=11, title_fontsize=12)

plt.tight_layout()
plt.show()

ANALISE PARTE 2

In [ ]:
import os
import pandas as pd
import numpy as np
import re
from sklearn.cluster import KMeans

pasta_features = 'Project_Features'

# 1. Carregar ficheiros
ficheiro_speech = 'Telejornal_RTP_Nov_20_speech.pkl'
ficheiro_ocr = 'Telejornal_RTP_Nov_20_ocr.pkl'

print(f" A iniciar Fusão Multimodal: {ficheiro_speech} + {ficheiro_ocr}")

# --- PREPARAR SPEECH ---
df_speech = pd.read_pickle(os.path.join(pasta_features, ficheiro_speech))
df_speech = df_speech.dropna(subset=['text_embedding', 'transcript'])
df_speech = df_speech.rename(columns={'time stamp': 'timestamp'})
df_speech = df_speech.sort_values('timestamp').reset_index(drop=True)

print(" A calcular Clusters de Áudio (K-Means)...")
matriz_embeddings = np.vstack(df_speech['text_embedding'].values)
kmeans = KMeans(n_clusters=8, random_state=42, n_init=10)
df_speech['id_cena_audio'] = kmeans.fit_predict(matriz_embeddings)

df_speech['nova_noticia'] = df_speech['id_cena_audio'] != df_speech['id_cena_audio'].shift(1)
df_speech.loc[0, 'nova_noticia'] = False

# --- PREPARAR OCR ---
print("A processar e limpar os dados Visuais (OCR)...")
df_ocr = pd.read_pickle(os.path.join(pasta_features, ficheiro_ocr))

# Extrair o segundo exato a partir da string "frame_XXXX.jpg"
df_ocr['timestamp'] = df_ocr['Frame'].str.extract(r'frame_(\d+)').astype(float)

# Função para extrair o texto de dentro da lista de dicionários
def limpar_ocr(dados_ocr):
    if isinstance(dados_ocr, list):
        # Extrai o valor 'text' de cada dicionário e junta com um " | "
        textos = [str(item.get('text', '')) for item in dados_ocr if 'text' in item]
        return ' | '.join(textos)
    return ""

df_ocr['texto_ecra'] = df_ocr['OCR'].apply(limpar_ocr)
df_ocr = df_ocr.sort_values('timestamp').reset_index(drop=True)

# --- A FUSÃO (Merge AsOf) ---
print("A cruzar Áudio e Texto de Ecrã (Time-Series Merge)...")

# Juntamos as duas tabelas com base no timestamp mais próximo (com tolerância de 3 segundos)
df_multimodal = pd.merge_asof(
    df_speech,
    df_ocr[['timestamp', 'texto_ecra']],
    on='timestamp',
    direction='nearest',
    tolerance=3.0
)

# --- RESULTADOS ---
transicoes = df_multimodal[df_multimodal['nova_noticia']]

print(f"\n Fusão Completa! Identificadas {len(transicoes)} transições de cena.")
print("Resultados da Fusão (Início da Cena vs Texto no Ecrã):\n")

resultado_final = transicoes[['timestamp', 'transcript', 'texto_ecra']]
resultado_final.columns = ['Tempo (s)', 'O que disseram (Speech)', 'O que estava escrito (OCR)']

# Aumentar a largura das colunas do Pandas para conseguirmos ler o texto todo
pd.set_option('display.max_colwidth', 100)
display(resultado_final.head(10))

In [ ]:
import os
import pandas as pd

pasta_features = 'Project_Features'
ficheiro_ocr_teste = 'Telejornal_RTP_Nov_20_ocr.pkl'

print(f"🔍 A inspecionar o ficheiro OCR: {ficheiro_ocr_teste}")
caminho_ocr = os.path.join(pasta_features, ficheiro_ocr_teste)

if os.path.exists(caminho_ocr):
    df_ocr_teste = pd.read_pickle(caminho_ocr)

    print("\n=== COLUNAS ENCONTRADAS NO OCR ===")
    df_ocr_teste.info()

    print("\n=== PRIMEIRAS LINHAS (AMOSTRA) ===")
    display(df_ocr_teste.head(2))
else:
    print(f"❌ O ficheiro {ficheiro_ocr_teste} não foi encontrado na pasta '{pasta_features}'.")

In [ ]:
import os
import pandas as pd
import numpy as np
import re
from sklearn.cluster import KMeans

# ==========================================
# 1. CONFIGURAÇÕES E DICIONÁRIOS
# ==========================================
pasta_features = 'Project_Features'
ficheiro_speech = 'Telejornal_RTP_Nov_20_speech.pkl'
ficheiro_ocr = 'Telejornal_RTP_Nov_20_ocr.pkl'

CANDIDATE_ALIASES = {
    "André Ventura": ["andré ventura", "ventura", "líder do chega"],
    "Cotrim Figueiredo": ["cotrim", "cotrim figueiredo", "joão cotrim de figueiredo"],
    "Luís Marques Mendes": ["marques mendes", "luís marques mendes"],
    "Henrique Gouveia e Melo": ["gouveia e melo", "henrique gouveia e melo"],
    "António José Seguro": ["antónio josé seguro", "josé seguro", "antónio seguro"],
    "António Filipe": ["antónio filipe"],
    "Catarina Martins": ["catarina martins", "ex coordenadora do bloco"],
    "Jorge Pinto": ["jorge pinto"]
}

PARTY_ALIASES = {
    "CHEGA": ["chega", "partido chega"],
    "IL": ["il", "iniciativa liberal", "liberais"],
    "PSD": ["psd", "ad", "aliança democrática", "partido social democrata"],
    "PS": ["ps", "partido socialista", "socialistas"],
    "PCP/CDU": ["pcp", "cdu", "partido comunista"],
    "BE": ["be", "bloco de esquerda"],
    "LIVRE": ["livre", "partido livre"],
    "CDS": ["cds", "cds pp", "centro democrático social"],
}

THEME_ALIASES = {
    "Eleições/Campanha": ["presidenciais", "eleições", "eleicao", "candidato", "candidatura", "campanha", "urna", "debate", "eleitor"],
    "Sondagens": ["sondagem", "sondagens", "barómetro", "voto", "intenção de voto", "projecção"],
    "Governo/Partidos": ["governo", "primeiro ministro", "montenegro", "parlamento", "partido", "oposição"],
    "Saúde": ["saúde", "sns", "hospital", "médico", "médica", "medica", "enfermeiro", "urgência", "inem", "ambulância"],
    "Economia": ["economia", "inflação", "preços", "imposto", "orçamento", "salário", "rendimento", "juros"],
    "Habitação": ["habitação", "casa", "arrendamento", "renda", "senhorio", "inquilino"],
    "Educação": ["educação", "escola", "professor", "aluno", "ensino", "creche"],
    "Justiça/Segurança": ["justiça", "tribunal", "polícia", "segurança", "crime", "corrupção", "caução", "burla", "suspeita", "operação", "suspensas", "pj"],
    "Internacional": ["ucrânia", "ucrania", "rússia", "guerra", "israel", "gaza", "palestina", "trump", "eua", "europa", "incêndio", "clima", "cop 30", "cimeira", "evacuado", "frança", "franceses", "casa branca", "porta-voz"],
    "Greves/Trabalho": ["greve", "sindicato", "trabalhador", "protesto", "patrões", "laboral", "legislação", "negociações", "lei laboral"]
}

# ==========================================
# 2. CORTES HÍBRIDOS (Speech + OCR)
# ==========================================
print("⚙️ A processar extração multimodal inteligente...")

df_speech = pd.read_pickle(os.path.join(pasta_features, ficheiro_speech)).dropna(subset=['text_embedding', 'transcript'])
df_speech = df_speech.rename(columns={'time stamp': 'timestamp'}).sort_values('timestamp')

n_clusters = min(20, len(df_speech) - 2)
matriz_embeddings = np.vstack(df_speech['text_embedding'].values)
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
df_speech['id_cena_audio'] = kmeans.fit_predict(matriz_embeddings)
cortes_audio = df_speech[df_speech['id_cena_audio'] != df_speech['id_cena_audio'].shift(1)]['timestamp'].tolist()

df_ocr = pd.read_pickle(os.path.join(pasta_features, ficheiro_ocr))
df_ocr['timestamp'] = df_ocr['Frame'].str.extract(r'frame_(\d+)').astype(float)
df_ocr['texto_ecra'] = df_ocr['OCR'].apply(lambda d: ' | '.join([str(i.get('text', '')) for i in d if 'text' in i]) if isinstance(d, list) else "")
df_ocr = df_ocr.sort_values('timestamp').reset_index(drop=True)

# Lógica de corte Visual
todas_palavras = set(sum([v for v in THEME_ALIASES.values()], []))
cortes_ocr = []
palavras_anteriores = set()
timestamp_ultimo_corte = 0

for idx, row in df_ocr.iterrows():
    texto = str(row['texto_ecra']).lower()
    palavras_atuais = {p for p in todas_palavras if re.search(rf'(?<!\w){re.escape(p)}(?!\w)', texto)}

    if palavras_atuais and len(palavras_atuais.intersection(palavras_anteriores)) == 0:
        if (row['timestamp'] - timestamp_ultimo_corte) > 20:
            cortes_ocr.append(row['timestamp'])
            timestamp_ultimo_corte = row['timestamp']
    if palavras_atuais: palavras_anteriores = palavras_atuais

todos_cortes = sorted(list(set([0.0] + cortes_audio + cortes_ocr)))
timestamps_finais = [todos_cortes[0]]
for t in todos_cortes[1:]:
    if (t - timestamps_finais[-1]) > 15: timestamps_finais.append(t)

# ==========================================
# 3. CLASSIFICAÇÃO(FILTRO DE CONTEXTO NLP)
# ==========================================
transicoes = pd.DataFrame({'timestamp': timestamps_finais})
transicoes = pd.merge_asof(transicoes, df_ocr[['timestamp', 'texto_ecra']], on='timestamp', direction='forward', tolerance=5.0)
transicoes = pd.merge_asof(transicoes, df_speech[['timestamp', 'transcript']], on='timestamp', direction='nearest', tolerance=30.0).fillna("")

transicoes['Tempo_Fim_s'] = transicoes['timestamp'].shift(-1)
transicoes.loc[transicoes.index[-1], 'Tempo_Fim_s'] = df_speech['timestamp'].max()
transicoes['Duracao_Segundos'] = transicoes['Tempo_Fim_s'] - transicoes['timestamp']

def classificar_tema_definitivo(ocr, transcricao):
    texto_ocr = str(ocr).lower()
    texto_raw = f"{ocr} {transcricao}".lower()

    # Filtro de falsos positivos (limpeza de expressões figurativas ou compostas)
    exclusoes = {
        "dar trabalho": "esforco",
        "bom trabalho": "esforco",
        "posto de trabalho": "emprego",
        "casa civil": "presidencia",
        "ordem dos médicos": "ordem_med"
    }
    for ruim, bom in exclusoes.items():
        texto_raw = texto_raw.replace(ruim, bom)
        texto_ocr = texto_ocr.replace(ruim, bom)

    encontrados = []

    # 1. Candidatos (Busca livre)
    for c, keys in CANDIDATE_ALIASES.items():
        if any(re.search(rf'(?<!\w){re.escape(k)}(?!\w)', texto_raw) for k in keys):
            encontrados.append(f"Cand: {c}")

    # 2. Partidos (Case-Sensitive para siglas)
    texto_original_misto = f"{ocr} {transcricao}"
    for p, keys in PARTY_ALIASES.items():
        for k in keys:
            if k.lower() in ['ps', 'psd', 'be', 'il', 'ad', 'cds', 'pcp', 'cdu']:
                if re.search(rf'(?<!\w){re.escape(k.upper())}(?!\w)', texto_original_misto): encontrados.append(f"Partido: {p}")
            elif k.lower() in ['chega', 'livre']:
                if re.search(rf'(?<!\w){k.capitalize()}(?!\w)', texto_original_misto) or re.search(rf'(?<!\w){k.upper()}(?!\w)', texto_original_misto) or f"partido {k.lower()}" in texto_raw:
                    encontrados.append(f"Partido: {p}")
            else:
                if re.search(rf'(?<!\w){re.escape(k.lower())}(?!\w)', texto_raw): encontrados.append(f"Partido: {p}")

    # 3. Temas
    palavras_perigosas = ['casa', 'casas', 'renda', 'rendas', 'voto', 'votos', 'urna', 'urnas', 'crime', 'crimes', 'aulas']

    for t, keys in THEME_ALIASES.items():
        for k in keys:
            k_lower = k.lower()
            if k_lower in palavras_perigosas:
                # Exige que esteja no OCR oficial
                if re.search(rf'(?<!\w){re.escape(k_lower)}(?!\w)', texto_ocr):
                    # Impede que "Casa Branca" acione a "Habitação" isoladamente
                    if k_lower == 'casa' and re.search(r'(?<!\w)casa branca(?!\w)', texto_ocr):
                        continue
                    encontrados.append(f"Tema: {t}")
            else:
                # Palavras fortes valem tanto no ecrã como na fala
                if re.search(rf'(?<!\w){re.escape(k_lower)}(?!\w)', texto_raw):
                    encontrados.append(f"Tema: {t}")

    encontrados = list(dict.fromkeys(encontrados)) # Remove duplicados
    return " + ".join(encontrados) if encontrados else "Outros/Geral"

transicoes['Classificacao'] = transicoes.apply(lambda row: classificar_tema_definitivo(row['texto_ecra'], row['transcript']), axis=1)

# ==========================================
# 4. AGRUPAMENTO DAS PEÇAS E TABELA FINAL
# ==========================================
transicoes['Mudar_Grupo'] = (transicoes['Classificacao'] != transicoes['Classificacao'].shift(1)).astype(int)
transicoes['ID_Agrupamento'] = transicoes['Mudar_Grupo'].cumsum()

relatorio_final = transicoes.groupby('ID_Agrupamento').agg(
    Timestamp_Inicio=('timestamp', 'min'),
    Categoria=('Classificacao', 'first'),
    Duracao_Total_Segundos=('Duracao_Segundos', 'sum'),
    Rodape_OCR=('texto_ecra', 'first')
).reset_index(drop=True)

# Limpeza e formatação de durações
relatorio_final = relatorio_final[relatorio_final['Duracao_Total_Segundos'] > 15].copy()
relatorio_final['Duração'] = relatorio_final['Duracao_Total_Segundos'].apply(lambda x: f"{int(x // 60)}m {int(x % 60)}s")
relatorio_final['Rodapé'] = relatorio_final['Rodape_OCR'].str.replace(r'<[^>]+>', '', regex=True).str[:75]

tabela_display = relatorio_final[['Timestamp_Inicio', 'Categoria', 'Duração', 'Rodapé']].reset_index(drop=True)

pd.set_option('display.max_colwidth', 90)
print("\n" + "="*95)
print("🎯 RESULTADO FINAL")
print("="*95)
display(tabela_display)

Speech rate dentro do telejornal
(pivot vs voz off)

In [ ]:
import os
import pandas as pd
import numpy as np
import re
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display

# ==========================================
# 1. CARREGAMENTO E ALINHAMENTO
# ==========================================
pasta_features = 'Project_Features'
ficheiro_speech = 'Telejornal_RTP_Nov_20_speech.pkl'
ficheiro_ocr = 'Telejornal_RTP_Nov_20_ocr.pkl'

PALAVRAS_PIVO = ['boa noite', 'reportagem', 'jornalista', 'em direto', 'trabalho de',
                 'vemos agora', 'em estúdio', 'destaque', 'acompanhar', 'conta-nos',
                 'veja', 'imagens']

print("⚙️ A carregar dados e a calcular quebras semânticas...")

df_speech = pd.read_pickle(os.path.join(pasta_features, ficheiro_speech)).dropna(subset=['transcript', 'text_embedding'])
df_speech['timestamp'] = df_speech['timestamp'].astype(float)
df_speech['duration'] = df_speech['duration'].astype(float)
df_speech['mid_time'] = df_speech['timestamp'] + (df_speech['duration'] / 2)
df_speech = df_speech.sort_values('timestamp').reset_index(drop=True)

# Calcular quebra de contexto (Embeddings)
embeddings = np.vstack(df_speech['text_embedding'].values)
sims = [1.0]
for i in range(1, len(embeddings)):
    sim = cosine_similarity(embeddings[i].reshape(1, -1), embeddings[i-1].reshape(1, -1))[0][0]
    sims.append(sim)
df_speech['sim_to_prev'] = sims

# Alinhar OCR
df_ocr = pd.read_pickle(os.path.join(pasta_features, ficheiro_ocr))
df_ocr['timestamp'] = df_ocr['Frame'].str.extract(r'frame_(\d+)').astype(float)
df_ocr['texto_ocr'] = df_ocr['OCR'].apply(lambda d: ' '.join([str(i.get('text', '')) for i in d if 'text' in i]) if isinstance(d, list) else "")

df_merged = pd.merge_asof(
    df_speech.sort_values('mid_time'),
    df_ocr[['timestamp', 'texto_ocr']].sort_values('timestamp'),
    left_on='mid_time',
    right_on='timestamp',
    suffixes=('', '_ocr'),
    direction='nearest',
    tolerance=2.0
).fillna("")

# ==========================================
# 2. SHATTER: ESTILHAÇAR O VÍDEO
# ==========================================
print("🔪 A estilhaçar o vídeo por silêncios e mudanças de tópico...")

df_merged['fim_anterior'] = (df_merged['timestamp'].shift(1) + df_merged['duration'].shift(1)).fillna(0)
df_merged['gap'] = df_merged['timestamp'] - df_merged['fim_anterior']

# Corta se houver pausa > 1.5s OU se o tópico mudar drasticamente (< 0.85 semelhança)
df_merged['corte'] = (df_merged['gap'] > 1.5) | (df_merged['sim_to_prev'] < 0.85)
df_merged.loc[0, 'corte'] = True
df_merged['id_bloco_bruto'] = df_merged['corte'].cumsum()

# ==========================================
# 3. AVALIAÇÃO DE SEGMENTOS CURTOS
# ==========================================
print("🧠 A avaliar os blocos com a Regra dos 90s e Ecrã Limpo...")
avaliacoes = []

for id_bloco, grupo in df_merged.groupby('id_bloco_bruto'):
    inicio = grupo['timestamp'].min()
    fim = grupo['timestamp'].max() + grupo['duration'].iloc[-1]
    duracao = fim - inicio

    if duracao < 3: continue # Ignorar ruídos muito curtos

    txt_fala = " ".join([str(t) for t in grupo['transcript'] if t]).lower()
    txt_ocr = " ".join([str(t) for t in grupo['texto_ocr'] if t]).lower()

    tem_palavras = any(re.search(rf'(?<!\w){kw}(?!\w)', txt_fala) for kw in PALAVRAS_PIVO)

    # Análise de OCR para desempatar:
    # Contamos palavras com mais de 3 letras no ecrã para ignorar ruído
    palavras_ecra = set([w for w in txt_ocr.split() if len(w) > 3])
    ecra_muito_limpo = len(palavras_ecra) < 12
    tem_tag_reportagem = any(kw in txt_ocr for kw in ['direto', 'repórter', 'enviado', 'polícia', 'tribunal'])

    is_pivo = False

    # Heurística Estrutural:
    if id_bloco == 1:
        is_pivo = True # A abertura é sempre o estúdio
    elif duracao < 95: # Se for maior que 95s, é automaticamente Reportagem
        if tem_palavras:
            is_pivo = True
        elif ecra_muito_limpo and not tem_tag_reportagem:
            # Se não usou palavras de passagem, mas o ecrã não tem rodapés de notícia, é o pivô
            is_pivo = True

    avaliacoes.append({
        'inicio': inicio,
        'fim': fim,
        'is_pivo': is_pivo,
        'duracao': duracao,
        'transcript': txt_fala
    })

df_avaliado = pd.DataFrame(avaliacoes)

# ==========================================
# 4. MERGE: COLAPSAR A TIMELINE
# ==========================================
print("📦 A fundir blocos semelhantes para construir a cronologia...")
df_avaliado['mudou_estado'] = df_avaliado['is_pivo'] != df_avaliado['is_pivo'].shift(1)
df_avaliado.loc[0, 'mudou_estado'] = True
df_avaliado['id_macro'] = df_avaliado['mudou_estado'].cumsum()

relatorio = []
for id_macro, grupo in df_avaliado.groupby('id_macro'):
    inicio = grupo['inicio'].min()
    fim = grupo['fim'].max()
    duracao = fim - inicio

    if duracao < 8: continue # Filtra artefactos resultantes do colapso

    evento = "Estúdio (Pivô)" if grupo['is_pivo'].iloc[0] else "Peça / Repórter"

    relatorio.append({
        'Início': f"{int(inicio // 60)}m {int(inicio % 60):02d}s",
        'Timestamp (s)': round(inicio, 1),
        'Duração': f"{int(duracao // 60)}m {int(duracao % 60):02d}s",
        'Evento': evento,
        'Transcrição (Amostra)': " ".join(grupo['transcript'])[:120].strip() + "..."
    })

df_final = pd.DataFrame(relatorio)

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_rows', 100)
print("\n" + "="*100)
print("🎯 RESULTADO ALÍNEA A (ARQUITETURA SHATTER & MERGE)")
print("="*100)
display(df_final)

In [ ]:
import os
import pandas as pd
from IPython.display import display

# ==========================================
# 1. CARREGAMENTO DOS DADOS
# ==========================================
pasta_features = 'Project_Features'
ficheiro_speech = 'Telejornal_RTP_Nov_20_speech.pkl'

df_speech = pd.read_pickle(os.path.join(pasta_features, ficheiro_speech)).dropna(subset=['transcript'])
df_speech['timestamp'] = df_speech['timestamp'].astype(float)
df_speech['duration'] = df_speech['duration'].astype(float)
df_speech = df_speech.sort_values('timestamp').reset_index(drop=True)

# ==========================================
# 2. CÁLCULO DOS SILÊNCIOS (GAPS)
# ==========================================
# Fim de cada intervenção
df_speech['end_time'] = df_speech['timestamp'] + df_speech['duration']

# Gap: Diferença entre o início atual e o fim da intervenção anterior
df_speech['fim_anterior'] = df_speech['end_time'].shift(1).fillna(0)
df_speech['gap'] = df_speech['timestamp'] - df_speech['fim_anterior']

# ==========================================
# 3. SEGMENTAÇÃO (LIMIAR: 2.0 SEGUNDOS)
# ==========================================
df_speech['is_cut'] = df_speech['gap'] > 2
df_speech.loc[0, 'is_cut'] = True
df_speech['block_id'] = df_speech['is_cut'].cumsum()

# ==========================================
# 4. GERAÇÃO DO RELATÓRIO COM TIMESTAMPS
# ==========================================
blocos = []
for block_id, grupo in df_speech.groupby('block_id'):
    inicio = grupo['timestamp'].min()
    fim = grupo['end_time'].max()
    duracao = fim - inicio

    # Gap que originou este bloco (o primeiro valor da janela)
    gap_originador = grupo['gap'].iloc[0] if block_id > 1 else 0.0

    texto = " ".join([str(t) for t in grupo['transcript'] if t])

    blocos.append({
        'Bloco': block_id,
        'Início': f"{int(inicio // 60):02d}m {int(inicio % 60):02d}s",
        'Timestamp (s)': round(inicio, 1),
        'Fim': f"{int(fim // 60):02d}m {int(fim % 60):02d}s",
        'Duração (s)': round(duracao, 1),
        'Silêncio (s)': round(gap_originador, 1),
        'Transcrição (Amostra)': texto[:120].strip() + "..."
    })

df_cortes = pd.DataFrame(blocos)

# Output
pd.set_option('display.max_colwidth', 150)
pd.set_option('display.max_rows', 100)
print("\n" + "="*120)
print("✂️ SEGMENTAÇÃO POR SILÊNCIOS (> 2.0 SEGUNDOS)")
print("="*120)
display(df_cortes)

Analise MultiModal (OCR + SPEECH + AUDIO)


In [ ]:
import os
import pandas as pd
import numpy as np
from pathlib import Path

# ==========================================
# 1. CARREGAMENTO DOS EXCEIS DE ÁUDIO
# ==========================================
df_seconds = pd.read_excel("speaker_by_second_all_debates.xlsx", sheet_name="Second_by_second")
df_instances = pd.read_excel("candidate_audio_embeddings.xlsx", sheet_name="Selected_Speaker_Instances")

# Mapa de oradores: ("Nome do Debate", "Person X") -> "Nome do Candidato"
speaker_map = df_instances.set_index(["debate_name", "local_person"])["candidate"].to_dict()

def map_to_candidate(row):
    person = row["estimated_speaker"]
    debate = row["debate_name"]
    return speaker_map.get((debate, person), "Moderador/Outro")

df_seconds["candidate"] = df_seconds.apply(map_to_candidate, axis=1)

# ==========================================
# 2. IMPORTAÇÃO E CRUZAMENTO DOS FICHEIROS SPEECH
# ==========================================
pasta_features = 'Project_Features'
DATA_DIR = Path(pasta_features)
speech_files = sorted(DATA_DIR.glob("*_speech.pkl"))

all_debate_transcripts = []

for speech_file in speech_files:
    filename = speech_file.name.replace("_speech.pkl", "")

    # Processar apenas os debates (ignorar telejornais)
    if "_vs_" not in filename:
        continue

    candidate_part, date_part = filename.split("_vs_")
    candidate_1 = candidate_part.replace("_", " ")
    candidate_2 = " ".join(date_part.split("_")[:-2])
    debate_name = f"{candidate_1} vs {candidate_2}"

    # Carregamento e limpeza (Lógica pré-definida)
    df_speech = pd.read_pickle(speech_file).dropna(subset=['transcript'])

    # Lidar com diferenças no nome da coluna de tempo (caso existam)
    time_col = 'time stamp' if 'time stamp' in df_speech.columns else 'timestamp'

    df_speech[time_col] = df_speech[time_col].astype(float)
    df_speech['duration'] = df_speech['duration'].astype(float)
    df_speech = df_speech.sort_values(time_col).reset_index(drop=True)

    # Calcular o ponto médio para cruzamento exato
    df_speech["midpoint_sec"] = (df_speech[time_col] + (df_speech["duration"] / 2)).astype(int)

    # Filtrar tabela de segundos para o debate atual
    debate_seconds = df_seconds[df_seconds["debate_name"] == debate_name].copy()

    # Cruzamento (Merge)
    df_merged = pd.merge(
        df_speech,
        debate_seconds[["second", "candidate"]],
        left_on="midpoint_sec",
        right_on="second",
        how="left"
    )

    df_merged["candidate"] = df_merged["candidate"].fillna("Desconhecido")
    df_merged["debate_name"] = debate_name

    # Padronizar o nome da coluna de tempo para facilitar análise posterior
    df_merged.rename(columns={time_col: 'timestamp'}, inplace=True)

    all_debate_transcripts.append(df_merged)

# ==========================================
# 3. CONSOLIDAÇÃO E EXPORTAÇÃO
# ==========================================
if all_debate_transcripts:
    df_final_transcripts = pd.concat(all_debate_transcripts, ignore_index=True)

    # Agregação do texto por candidato (excluindo moderador)
    text_by_candidate = (
        df_final_transcripts[df_final_transcripts["candidate"] != "Moderador/Outro"]
        .groupby(["candidate", "debate_name"])["transcript"]
        .apply(lambda x: " ".join(x.astype(str)))
        .reset_index()
    )

    # Guardar os resultados
    text_by_candidate.to_excel("transcricoes_completas_por_candidato.xlsx", index=False)
    df_final_transcripts.to_excel("frases_detalhadas_por_candidato.xlsx", index=False)

In [ ]:
import pandas as pd
import os
import re
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path


# Dicionários de Temas
THEME_ALIASES = {

    "Eleições/Campanha": [
        "presidenciais", "presidencial",
        "eleições", "eleicoes", "eleição", "eleicao",
        "eleições presidenciais", "eleicoes presidenciais",
        "candidato", "candidata", "candidatos", "candidaturas",
        "campanha", "campanha eleitoral",
        "debate eleitoral", "debates eleitorais",
        "voto", "votos", "eleitores", "urna", "urnas"
    ],

    "Sondagens": [
        "sondagem", "sondagens",
        "barómetro", "barometro",
        "intenção de voto", "intencao de voto",
        "intenções de voto", "intencoes de voto",
        "estimativa eleitoral", "projeção eleitoral", "projecao eleitoral",
        "pontos percentuais"
    ],

    "Governo/Partidos": [
        "governo", "executivo",
        "primeiro ministro", "primeiro-ministro",
        "ministro", "ministra", "ministros",
        "parlamento", "assembleia da república", "assembleia da republica",
        "oposição", "oposicao",
        "líder parlamentar", "lider parlamentar",
        "grupo parlamentar", "maioria absoluta",
        "moção de censura", "mocao de censura",
        "partidos políticos", "partidos politicos"
    ],

    "Saúde": [
        "saúde", "saude",
        "sns", "serviço nacional de saúde", "servico nacional de saude",
        "hospital", "hospitais",
        "médico", "medico", "médicos", "medicos",
        "enfermeiro", "enfermeira", "enfermeiros", "enfermeiras",
        "urgência", "urgencia", "urgências", "urgencias",
        "doente", "doentes", "utente", "utentes",
        "lista de espera", "listas de espera"
    ],

    "Economia/Proteção Social": [
        "economia", "económico", "economico",
        "inflação", "inflacao",
        "preços", "precos", "custo de vida",
        "impostos", "irs", "iva", "irc",
        "orçamento", "orcamento", "orçamento do estado", "orcamento do estado",
        "défice", "defice", "dívida pública", "divida publica",
        "salário", "salario", "salários", "salarios",
        "salário mínimo", "salario minimo",
        "pensões", "pensoes", "reformas",
        "segurança social", "seguranca social",
        "subsídio", "subsidio", "subsídios", "subsidios",
        "apoios sociais", "apoio social",
        "empresas", "juros", "taxas de juro",
        "banco de portugal", "bce", "bancos"
    ],

    "Habitação": [
        "habitação", "habitacao",
        "arrendamento", "arrendar",
        "renda", "rendas",
        "senhorio", "senhorios",
        "inquilino", "inquilinos",
        "crédito habitação", "credito habitacao",
        "empréstimo da casa", "emprestimo da casa",
        "preço das casas", "precos das casas",
        "mercado imobiliário", "mercado imobiliario",
        "imobiliário", "imobiliario"
    ],

    "Educação": [
        "educação", "educacao",
        "escola", "escolas",
        "professor", "professores",
        "aluno", "alunos",
        "ensino", "aulas",
        "creche", "creches",
        "universidade", "universidades",
        "estudante", "estudantes",
        "exames nacionais", "ano letivo", "ano lectivo"
    ],

    "Justiça/Segurança": [
        "justiça", "justica",
        "tribunal", "tribunais",
        "polícia", "policia",
        "psp", "gnr", "pj", "polícia judiciária", "policia judiciaria",
        "crime", "crimes",
        "fraude", "fraudes",
        "sócrates", "socrates",
        "josé sócrates", "jose socrates",
        "homicídio", "homicidio",
        "agressão", "agressao",
        "corrupção", "corrupcao",
        "pgr", "ministério público", "ministerio publico",
        "detido", "detidos", "arguido", "arguidos",
        "prisão", "prisao",
        "caução", "caucao",
        "buscas", "operação policial", "operacao policial",

        # Acidentes e segurança pública
        "acidente", "acidentes",
        "colisão", "colisao",
        "despiste",
        "queda",
        "descarrilamento",
        "vítima", "vitima", "vítimas", "vitimas",
        "ferido", "feridos", "ferida", "feridas",
        "morto", "mortos", "morta", "mortas",
        "falha de segurança", "falha de seguranca",
        "falha técnica", "falha tecnica",
        "investigação", "investigacao",
        "inquérito", "inquerito"
    ],

    "Internacional": [
        "ucrânia", "ucrania",
        "rússia", "russia",
        "guerra na ucrânia", "guerra na ucrania",
        "israel", "gaza", "palestina", "hamas",
        "médio oriente", "medio oriente",
        "trump", "donald trump",
        "casa branca",
        "eua", "estados unidos",
        "bruxelas", "união europeia", "uniao europeia",
        "diplomacia", "diplomacia europeia",
        "diplomático", "diplomatico",
        "diplomática", "diplomatica",
        "relações diplomáticas", "relacoes diplomaticas",
        "nato", "onu",
        "frança", "franca",
        "espanha", "brasil", "china",
        "reino unido", "alemanha",
        "áfrica do sul", "africa do sul",
        "g20", "g7",

        # Política norte-americana
        "republicanos", "republicano",
        "partido republicano",
        "democratas", "democrata",
        "partido democrata",
        "congresso americano",
        "senado americano",
        "câmara dos representantes", "camara dos representantes"
    ],

    "Greves/Trabalho": [
        "greve", "greves",
        "sindicato", "sindicatos",
        "trabalhador", "trabalhadores",
        "protesto", "protestos",
        "manifestação", "manifestacao",
        "manifestantes",
        "contrato coletivo", "contrato colectivo",
        "concertação social", "concertacao social"
    ],

    "Transportes/Mobilidade": [
        "transportes",
        "metro", "metropolitano",
        "comboio", "comboios",
        "cp", "fertagus",
        "autocarro", "autocarros",
        "trânsito", "transito",
        "aeroporto",
        "tap", "tap air portugal",
        "avião", "aviao", "aviões", "avioes",
        "estrada", "autoestrada",
        "portagens",

        # Operadores e transportes urbanos
        "carris",
        "elétrico", "eletrico",
        "ascensor", "ascensores",
        "funicular",

        # Calçada/Elevador da Glória
        "calçada da glória", "calcada da gloria",
        "elevador da glória", "elevador da gloria"
    ],

    "Ambiente/Meteorologia/Proteção Civil": [
        "ambiente",
        "clima", "climático", "climatico",
        "alterações climáticas", "alteracoes climaticas",
        "seca", "chuva", "temporal",
        "inundações", "inundacoes", "cheias",
        "emissões", "emissoes", "poluição", "poluicao",
        "meteorologia", "previsão meteorológica", "previsao meteorologica",
        "temperatura", "vento", "frio", "calor", "neve",
        "incêndio", "incendio", "incêndios", "incendios",
        "bombeiros",
        "proteção civil", "protecao civil",
        "chamas", "evacuação", "evacuacao"
    ],

    "Desporto": [
        "futebol",
        "benfica", "sporting", "fc porto",
        "liga dos campeões", "liga dos campeoes",
        "liga portuguesa",
        "campeonato nacional",
        "seleção nacional", "selecao nacional",
        "treinador", "jogador", "jogadores",
        "golo", "golos",
        "estádio", "estadio"
    ],

    "Cultura": [
        "cultura",
        "cinema", "teatro",
        "música", "musica",
        "festival", "festivais",
        "livro", "livros",
        "exposição", "exposicao",
        "museu", "museus",
        "artista", "artistas",
        "concerto", "concertos"
    ],
}

# Mapeamento de Partidos
CANDIDATE_PARTY = {
    "Ventura": "CHEGA",
    "Cotrim Figueiredo": "IL",
    "Marques Mendes": "PSD + CDS",
    "Gouveia Melo": "Independente",
    "Seguro": "PS",
    "Filipe": "PCP",
    "Martins": "BE",
    "Pinto": "LIVRE"
}

# 1. Carregar dados
df = pd.read_excel('transcricoes_completas_por_candidato.xlsx')

# 2. Função de contagem exata (regex com \b para evitar falsos positivos)
def count_themes(text):
    text = str(text).lower()
    counts = {theme: 0 for theme in THEME_ALIASES}
    for theme, keywords in THEME_ALIASES.items():
        for kw in keywords:
            pattern = rf'\b{re.escape(kw)}\b'
            counts[theme] += len(re.findall(pattern, text))
    return pd.Series(counts)

# 3. Aplicar contagem
theme_counts = df['transcript'].apply(count_themes)
df_themes = pd.concat([df[['candidate']], theme_counts], axis=1)

# 4. Agregar por candidato e mapear partido
candidate_themes = df_themes.groupby('candidate').sum()
candidate_themes['Partido'] = candidate_themes.index.map(lambda x: CANDIDATE_PARTY.get(x, "Outro"))

# Reordenar colunas
cols = ['Partido'] + [c for c in candidate_themes.columns if c != 'Partido']
candidate_themes = candidate_themes[cols]

# 5. Normalizar para obter Perfil Temático (Percentagem)
theme_only = candidate_themes.drop(columns=['Partido'])
candidate_profiles = theme_only.div(theme_only.sum(axis=1), axis=0) * 100
candidate_profiles = candidate_profiles.fillna(0)

# Adicionar Partido ao índice para o gráfico
candidate_profiles.index = [f"{cand} ({CANDIDATE_PARTY.get(cand, 'Outro')})" for cand in candidate_profiles.index]

# 6. Gerar Heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(candidate_profiles, annot=True, cmap='YlGnBu', fmt='.1f', linewidths=.5, cbar_kws={'label': '% de Foco Temático'})
plt.title('Perfil Temático dos Candidatos nos Debates (%)', pad=20, fontsize=14)
plt.ylabel('Candidato (Partido)')
plt.xlabel('Temas')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('heatmap_temas.png', dpi=300)
plt.show()

# Imprimir Tabela de Frequências Absolutas
print(candidate_themes)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# ==========================================
# 1. ÂNCORAS PARTIDÁRIAS (Sedes Ideológicas)
# ==========================================
TARGET_TEMPLATES = {
    "BE": {"X": -1.5, "Y": 1.0},
    "LIVRE": {"X": -1.5, "Y": 1.2},
    "PCP": {"X": -2.0, "Y": 0.0},
    "PS": {"X": -0.1, "Y": 0.2},
    "IL": {"X": 1.6, "Y": 1.3},
    "PSD": {"X": 1.5, "Y": -0.8},
    "CHEGA": {"X": 0.5, "Y": -1.8},
    "Independente": {"X": 0.0, "Y": 0.0}
}

# ==========================================
# 2. CÁLCULO MODERADO DO DESVIO
# ==========================================
w_X = {
    'Greves/Trabalho': -1.0, 'Habitação': -1.0, 'Saúde': -0.5, 'Educação': -0.5,
    'Justiça/Segurança': 0.5, 'Governo/Partidos': 0.2, 'Economia/Proteção Social': 0.8
}

w_Y = {
    'Cultura': 1.0, 'Ambiente/Meteorologia/Proteção Civil': 1.0, 'Educação': 0.5, 'Internacional': 0.5,
    'Justiça/Segurança': -1.0, 'Governo/Partidos': -0.2, 'Economia/Proteção Social': -0.2
}

def calc_axis(profile, weights):
    return sum(profile[theme] * w for theme, w in weights.items() if theme in profile)

compass_df = candidate_profiles.copy()
avg_profile = candidate_profiles.mean(axis=0)

# Calcular a força do desvio de cada candidato face à média do debate
compass_df['Delta_X'] = candidate_profiles.apply(lambda row: calc_axis(row - avg_profile, w_X), axis=1)
compass_df['Delta_Y'] = candidate_profiles.apply(lambda row: calc_axis(row - avg_profile, w_Y), axis=1)

# FATOR DE MODERAÇÃO: Dividir por 25 (suaviza os elásticos para não fugirem do gráfico)
fator_moderacao = 25
compass_df['Final_X'] = compass_df.index.map(lambda c: TARGET_TEMPLATES[CANDIDATE_PARTY[c]]['X']) + (compass_df['Delta_X'] / fator_moderacao)
compass_df['Final_Y'] = compass_df.index.map(lambda c: TARGET_TEMPLATES[CANDIDATE_PARTY[c]]['Y']) + (compass_df['Delta_Y'] / fator_moderacao)

# ==========================================
# 3. VISUALIZAÇÃO DA BÚSSOLA
# ==========================================
plt.figure(figsize=(12, 12))
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'

colors_map = {
    "CHEGA": "#2A2A54", "IL": "#00AEEF", "PSD": "#FF9900", "PS": "#FF66FF",
    "BE": "#CC0000", "PCP": "#FF0000", "LIVRE": "#99CC33", "Independente": "#808080"
}

# Preenchimento de Quadrantes
plt.axvspan(0, 4, ymin=0.5, ymax=1, alpha=0.03, color='blue')
plt.axvspan(-4, 0, ymin=0.5, ymax=1, alpha=0.03, color='green')
plt.axvspan(0, 4, ymin=0, ymax=0.5, alpha=0.03, color='purple')
plt.axvspan(-4, 0, ymin=0, ymax=0.5, alpha=0.03, color='red')

plt.axhline(0, color='#666666', linestyle='--', linewidth=1.2)
plt.axvline(0, color='#666666', linestyle='--', linewidth=1.2)

# Plotar as SEDES e Ligar as Linhas
for party, pos in TARGET_TEMPLATES.items():
    color = colors_map[party]
    plt.scatter(pos['X'], pos['Y'], color=color, s=150, marker='s', edgecolor='black', alpha=0.5, linewidth=1.2, zorder=4)
    plt.text(pos['X'] + 0.05, pos['Y'] - 0.08, f"Partido {party}", fontsize=9, style='italic', color='#444444', zorder=4)

    for cand, row in compass_df.iterrows():
        cand_name = cand.split(" (")[0] if " (" in cand else cand
        if CANDIDATE_PARTY.get(cand_name) == party and party != "Independente":
            plt.plot([row['Final_X'], pos['X']], [row['Final_Y'], pos['Y']], color=color, linestyle=':', alpha=0.6, linewidth=1.5)

# Plotar os CANDIDATOS
for cand, row in compass_df.iterrows():
    cand_name = cand.split(" (")[0] if " (" in cand else cand
    partido = CANDIDATE_PARTY.get(cand_name, "Outro")
    color = colors_map.get(partido, "#000000")

    plt.scatter(row['Final_X'], row['Final_Y'], color=color, s=280, edgecolor='black', linewidth=1.5, zorder=5)
    plt.text(row['Final_X'] + 0.06, row['Final_Y'] + 0.06, cand_name, fontsize=11, fontweight='bold', zorder=6)

# Títulos e Etiquetas Exatas
plt.text(0, 3.0, 'LIBERAL / PROGRESSISTA', ha='center', va='bottom', fontsize=12, fontweight='bold', color='green')
plt.text(0, -3.0, 'CONSERVADOR / AUTORIDADE', ha='center', va='top', fontsize=12, fontweight='bold', color='darkred')
plt.text(3.0, 0, 'DIREITA\nECONÓMICA', ha='left', va='center', fontsize=12, fontweight='bold', color='darkblue')
plt.text(-3.0, 0, 'ESQUERDA\nECONÓMICA', ha='right', va='center', fontsize=12, fontweight='bold', color='darkred')

plt.title('Bússola Multimodal: Distância entre Candidatos e Matrizes Partidárias', fontsize=15, pad=25, fontweight='bold')

plt.xlim(-3.2, 3.2)
plt.ylim(-3.2, 3.2)
plt.grid(True, which='both', linestyle=':', alpha=0.3)
plt.tight_layout()
plt.savefig('bussola_politica_final_moderada.png', dpi=300)
plt.show()

In [ ]:
import pandas as pd
import os

filepath = 'Project_Features/polls_timeline.pkl'

if os.path.exists(filepath):
    df_polls = pd.read_pickle(filepath)

    print("--- INFORMAÇÃO DO DATASET DE SONDAGENS ---")
    print(f"Número de Linhas e Colunas: {df_polls.shape}\n")
    print("Tipos de Dados:")
    print(df_polls.dtypes)
    print("\nPrimeiras 5 linhas:")
    print(df_polls.head())

    # Identificar a coluna de candidatos/partidos
    colunas_texto = df_polls.select_dtypes(include=['object', 'string', 'category']).columns
    for col in colunas_texto:
        print(f"\nValores únicos na coluna '{col}':")
        print(df_polls[col].unique()[:15])
else:
    print(f"File not found: {filepath}")
    # Check if the folder exists and what's in it
    if os.path.exists('Project_Features'):
        print("Files in Project_Features:")
        print([f for f in os.listdir('Project_Features') if 'poll' in f.lower() or 'pkl' in f.lower()][:10])
    else:
        print("Folder Project_Features not found. Current directory files:")
        print(os.listdir('.'))

In [ ]:
import pandas as pd
import os
import re
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Dicionários de Temas
THEME_ALIASES = {
    "Eleições/Campanha": ["presidenciais", "presidencial", "eleições", "eleicoes", "eleição", "eleicao", "eleições presidenciais", "eleicoes presidenciais", "candidato", "candidata", "candidatos", "candidaturas", "candidatura", "campanha", "campanha eleitoral", "debate", "debate eleitoral", "debates eleitorais", "voto", "votos", "eleitores", "urna", "urnas", "segunda", "segunda volta"],
    "Sondagens": ["sondagem", "sondagens", "barómetro", "barometro", "intenção de voto", "intencao de voto", "intenções de voto", "intencoes de voto", "estimativa eleitoral", "projeção", "projeção eleitoral", "projecao eleitoral", "pontos percentuais"],
    "Governo/Partidos": ["governo", "executivo", "primeiro ministro", "primeiro-ministro", "ministro", "ministra", "ministros", "parlamento", "assembleia", "assembleia da república", "assembleia da republica", "oposição", "oposicao", "líder", "líder parlamentar", "lider parlamentar", "grupo parlamentar", "maioria", "maioria absoluta", "moção de censura", "mocao de censura", "partidos", "partidos políticos", "partidos politicos", "política", "posição", "político", "direita", "psd", "sistema", "proposta", "poder"],
    "Saúde": ["saúde", "saude", "sns", "serviço nacional de saúde", "servico nacional de saude", "hospital", "hospitais", "médico", "medico", "médicos", "medicos", "enfermeiro", "enfermeira", "enfermeiros", "enfermeiras", "urgência", "urgencia", "urgências", "urgencias", "doente", "doentes", "utente", "utentes", "lista de espera", "listas de espera", "acesso", "serviços"],
    "Economia/Proteção Social": ["economia", "económico", "economico", "inflação", "inflacao", "preços", "precos", "custo de vida", "impostos", "irs", "iva", "irc", "orçamento", "orcamento", "orçamento do estado", "orcamento do estado", "défice", "defice", "dívida", "dívida pública", "divida publica", "salário", "salario", "salários", "salarios", "salário mínimo", "salario minimo", "pensões", "pensoes", "reformas", "segurança social", "seguranca social", "social", "subsídio", "subsidio", "subsídios", "subsidios", "apoios", "apoios sociais", "apoio social", "empresas", "empresa", "juros", "taxas de juro", "banco", "banco de portugal", "bce", "bancos", "jovens", "euros", "milhões", "números"],
    "Habitação": ["habitação", "habitacao", "arrendamento", "arrendar", "renda", "rendas", "senhorio", "senhorios", "inquilino", "inquilinos", "crédito", "credito", "crédito habitação", "credito habitacao", "empréstimo", "empréstimo da casa", "emprestimo da casa", "preço das casas", "precos das casas", "mercado imobiliário", "mercado imobiliario", "imobiliário", "imobiliario"],
    "Educação": ["educação", "educacao", "escola", "escolas", "professor", "professores", "aluno", "alunos", "ensino", "aulas", "creche", "creches", "universidade", "universidades", "estudante", "estudantes", "exames", "exames nacionais", "ano letivo", "ano lectivo"],
    "Justiça/Segurança": ["justiça", "justica", "tribunal", "tribunais", "polícia", "policia", "psp", "gnr", "pj", "polícia judiciária", "policia judiciaria", "crime", "crimes", "fraude", "fraudes", "sócrates", "socrates", "josé sócrates", "jose socrates", "homicídio", "homicidio", "agressão", "agressao", "corrupção", "corrupcao", "pgr", "público", "ministério público", "ministerio publico", "detido", "detidos", "arguido", "arguidos", "prisão", "prisao", "caução", "caucao", "buscas", "operação", "operação policial", "operacao policial", "acidente", "acidentes", "colisão", "colisao", "despiste", "queda", "descarrilamento", "vítima", "vitima", "vítimas", "vitimas", "ferido", "feridos", "ferida", "feridas", "morto", "mortos", "morta", "mortas", "falha de segurança", "falha de seguranca", "falha técnica", "falha tecnica", "investigação", "investigacao", "inquérito", "inquerito", "processo", "advogado", "violência", "caso", "lei"],
    "Internacional": ["ucrânia", "ucrania", "rússia", "russia", "guerra", "guerra na ucrânia", "guerra na ucrania", "israel", "gaza", "palestina", "hamas", "médio oriente", "medio oriente", "trump", "donald", "donald trump", "casa branca", "eua", "estados unidos", "bruxelas", "união", "união europeia", "uniao europeia", "europa", "europeia", "diplomacia", "diplomacia europeia", "diplomático", "diplomatico", "diplomática", "diplomatica", "relações diplomáticas", "relacoes diplomaticas", "nato", "onu", "frança", "franca", "espanha", "brasil", "china", "reino unido", "alemanha", "áfrica do sul", "africa do sul", "g20", "g7", "republicanos", "republicano", "partido republicano", "democratas", "democrata", "partido democrata", "congresso", "congresso americano", "senado americano", "câmara dos representantes", "camara dos representantes", "paz", "venezuela", "regime", "acordo", "segurança"],
    "Greves/Trabalho": ["greve", "greves", "sindicato", "sindicatos", "trabalhador", "trabalhadores", "trabalho", "protesto", "protestos", "manifestação", "manifestacao", "manifestantes", "contrato", "contrato coletivo", "contrato colectivo", "concertação", "concertação social", "concertacao social", "laboral", "direitos"],
    "Transportes/Mobilidade": ["transportes", "metro", "metropolitano", "comboio", "comboios", "cp", "fertagus", "autocarro", "autocarros", "trânsito", "transito", "aeroporto", "tap", "tap air portugal", "avião", "aviao", "aviões", "avioes", "estrada", "autoestrada", "portagens", "carris", "elétrico", "eletrico", "ascensor", "ascensores", "funicular", "calçada da glória", "calcada da gloria", "elevador da glória", "elevador da gloria"],
    "Ambiente/Meteorologia": ["ambiente", "clima", "climático", "climatico", "alterações climáticas", "alteracoes climaticas", "seca", "chuva", "temporal", "inundações", "inundacoes", "cheias", "emissões", "emissoes", "poluição", "poluicao", "meteorologia", "previsão meteorológica", "previsao meteorologica", "temperatura", "vento", "frio", "calor", "neve", "incêndio", "incendio", "incêndios", "incendios", "bombeiros", "proteção civil", "protecao civil", "chamas", "evacuação", "evacuacao"],
    "Desporto": ["futebol", "benfica", "sporting", "fc porto", "liga dos campeões", "liga dos campeoes", "liga portuguesa", "campeonato nacional", "seleção nacional", "selecao nacional", "treinador", "jogador", "jogadores", "golo", "golos", "estádio", "estadio"],
    "Cultura": ["cultura", "cinema", "teatro", "música", "musica", "festival", "festivais", "livro", "livros", "exposição", "exposicao", "museu", "museus", "artista", "artistas", "concerto", "concertos"]
}

# Mapeamento de Partidos
CANDIDATE_PARTY = {
    "Ventura": "CHEGA",
    "Cotrim Figueiredo": "IL",
    "Marques Mendes": "PSD + CDS",
    "Gouveia Melo": "Independente",
    "Seguro": "PS",
    "Filipe": "PCP",
    "Martins": "BE",
    "Pinto": "LIVRE"
}

# 1. Carregar dados ao nível do segmento
df_detalhado = pd.read_excel('frases_detalhadas_por_candidato.xlsx')

# Filtrar moderador
df_candidatos = df_detalhado[df_detalhado['candidate'] != 'Moderador/Outro'].copy()

# 2. Função de contagem exata
def count_themes(text):
    text = str(text).lower()
    counts = {theme: 0 for theme in THEME_ALIASES}
    for theme, keywords in THEME_ALIASES.items():
        for kw in keywords:
            pattern = rf'\b{re.escape(kw)}\b'
            counts[theme] += len(re.findall(pattern, text))
    return pd.Series(counts)

# 3. Aplicar contagem a cada frase individual
theme_counts = df_candidatos['transcript'].apply(count_themes)

# Obter tema dominante por segmento
def get_dominant_theme(row):
    if row.sum() == 0:
        return "Nenhum/Não Identificado"
    return row.idxmax()

df_candidatos['Tema_Dominante'] = theme_counts.apply(get_dominant_theme, axis=1)

# Guardar dataframe com os segmentos classificados
df_final_segmentos = pd.concat([df_candidatos, theme_counts], axis=1)
df_final_segmentos.to_csv('analise_temas_por_segmento.csv', index=False)

# 4. Agregação para visualização: Contar o número de segmentos em que o tema é dominante
segment_counts = df_candidatos[df_candidatos['Tema_Dominante'] != 'Nenhum/Não Identificado'].groupby(['candidate', 'Tema_Dominante']).size().unstack(fill_value=0)

# Adicionar colunas em falta para manter a consistência com o dicionário
for theme in THEME_ALIASES:
    if theme not in segment_counts.columns:
        segment_counts[theme] = 0

# Normalizar para obter percentagem de intervenções por tema
segment_profiles = segment_counts.div(segment_counts.sum(axis=1), axis=0) * 100
segment_profiles = segment_profiles.fillna(0)

# Adicionar Partido ao índice
segment_profiles.index = [f"{cand} ({CANDIDATE_PARTY.get(cand, 'Outro')})" for cand in segment_profiles.index]

# Filtrar apenas temas que têm pelo menos alguma ocorrência para simplificar o Heatmap
segment_profiles = segment_profiles.loc[:, (segment_profiles != 0).any(axis=0)]

# 5. Gerar Heatmap Segmentado
plt.figure(figsize=(14, 8))
sns.heatmap(segment_profiles, annot=True, cmap='YlGnBu', fmt='.1f', linewidths=.5, cbar_kws={'label': '% de Segmentos Dominados pelo Tema'})
plt.title('Perfil Temático por Intervenção/Segmento de Fala (%)', pad=20, fontsize=14)
plt.ylabel('Candidato (Partido)')
plt.xlabel('Tema Dominante no Segmento')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('heatmap_segmentos.png', dpi=300)
plt.show()

print("Processamento concluído. O ficheiro 'analise_temas_por_segmento.csv' foi gerado.")

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# ==========================================
# 1. CONFIGURAÇÕES E DICIONÁRIOS
# ==========================================
THEME_ALIASES = {
    "Eleições/Campanha": ["presidenciais", "presidencial", "eleições", "eleicoes", "eleição", "eleicao", "eleições presidenciais", "eleicoes presidenciais", "candidato", "candidata", "candidatos", "candidaturas", "campanha", "campanha eleitoral", "debate eleitoral", "debates eleitorais", "voto", "votos", "eleitores", "urna", "urnas"],
    "Sondagens": ["sondagem", "sondagens", "barómetro", "barometro", "intenção de voto", "intencao de voto", "intenções de voto", "intencoes de voto", "estimativa eleitoral", "projeção eleitoral", "projecao eleitoral", "pontos percentuais"],
    "Governo/Partidos": ["governo", "executivo", "primeiro ministro", "primeiro-ministro", "ministro", "ministra", "ministros", "parlamento", "assembleia da república", "assembleia da republica", "oposição", "oposicao", "líder parlamentar", "lider parlamentar", "grupo parlamentar", "maioria absoluta", "moção de censura", "mocao de censura", "partidos políticos", "partidos politicos"],
    "Saúde": ["saúde", "saude", "sns", "serviço nacional de saúde", "servico nacional de saude", "hospital", "hospitais", "médico", "medico", "médicos", "medicos", "enfermeiro", "enfermeira", "enfermeiros", "enfermeiras", "urgência", "urgencia", "urgências", "urgencias", "doente", "doentes", "utente", "utentes", "lista de espera", "listas de espera"],
    "Economia/Proteção Social": ["economia", "económico", "economico", "inflação", "inflacao", "preços", "precos", "custo de vida", "impostos", "irs", "iva", "irc", "orçamento", "orcamento", "orçamento do estado", "orcamento do estado", "défice", "defice", "dívida pública", "divida publica", "salário", "salario", "salários", "salarios", "salário mínimo", "salario minimo", "pensões", "pensoes", "reformas", "segurança social", "seguranca social", "subsídio", "subsidio", "subsídios", "subsidios", "apoios sociais", "apoio social", "empresas", "juros", "taxas de juro", "banco de portugal", "bce", "bancos"],
    "Habitação": ["habitação", "habitacao", "arrendamento", "arrendar", "renda", "rendas", "senhorio", "senhorios", "inquilino", "inquilinos", "crédito habitação", "credito habitacao", "empréstimo da casa", "emprestimo da casa", "preço das casas", "precos das casas", "mercado imobiliário", "mercado imobiliario", "imobiliário", "imobiliario"],
    "Educação": ["educação", "educacao", "escola", "escolas", "professor", "professores", "aluno", "alunos", "ensino", "aulas", "creche", "creches", "universidade", "universidades", "estudante", "estudantes", "exames nacionais", "ano letivo", "ano lectivo"],
    "Justiça/Segurança": ["justiça", "justica", "tribunal", "tribunais", "polícia", "policia", "psp", "gnr", "pj", "polícia judiciária", "policia judiciaria", "crime", "crimes", "fraude", "fraudes", "sócrates", "socrates", "josé sócrates", "jose socrates", "homicídio", "homicidio", "agressão", "agressao", "corrupção", "corrupcao", "pgr", "ministério público", "ministerio publico", "detido", "detidos", "arguido", "arguidos", "prisão", "prisao", "caução", "caucao", "buscas", "operação policial", "operacao policial", "acidente", "acidentes", "colisão", "colisao", "despiste", "queda", "descarrilamento", "vítima", "vitima", "vítimas", "vitimas", "ferido", "feridos", "ferida", "feridas", "morto", "mortos", "morta", "mortas", "falha de segurança", "falha de seguranca", "falha técnica", "falha tecnica", "investigação", "investigacao", "inquérito", "inquerito"],
    "Internacional": ["ucrânia", "ucrania", "rússia", "russia", "guerra na ucrânia", "guerra na ucrania", "israel", "gaza", "palestina", "hamas", "médio oriente", "medio oriente", "trump", "donald trump", "casa branca", "eua", "estados unidos", "bruxelas", "união europeia", "uniao europeia", "diplomacia", "diplomacia europeia", "diplomático", "diplomatico", "diplomática", "diplomatica", "relações diplomáticas", "relacoes diplomaticas", "nato", "onu", "frança", "franca", "espanha", "brasil", "china", "reino unido", "alemanha", "áfrica do sul", "africa do sul", "g20", "g7", "republicanos", "republicano", "partido republicano", "democratas", "democrata", "partido democrata", "congresso americano", "senado americano", "câmara dos representantes", "camara dos representantes"],
    "Greves/Trabalho": ["greve", "greves", "sindicato", "sindicatos", "trabalhador", "trabalhadores", "protesto", "protestos", "manifestação", "manifestacao", "manifestantes", "contrato coletivo", "contrato colectivo", "concertação social", "concertacao social"],
    "Transportes/Mobilidade": ["transportes", "metro", "metropolitano", "comboio", "comboios", "cp", "fertagus", "autocarro", "autocarros", "trânsito", "transito", "aeroporto", "tap", "tap air portugal", "avião", "aviao", "aviões", "avioes", "estrada", "autoestrada", "portagens", "carris", "elétrico", "eletrico", "ascensor", "ascensores", "funicular", "calçada da glória", "calcada da gloria", "elevador da glória", "elevador da gloria"],
    "Ambiente/Meteorologia/Proteção Civil": ["ambiente", "clima", "climático", "climatico", "alterações climáticas", "alteracoes climaticas", "seca", "chuva", "temporal", "inundações", "inundacoes", "cheias", "emissões", "emissoes", "poluição", "poluicao", "meteorologia", "previsão meteorológica", "previsao meteorologica", "temperatura", "vento", "frio", "calor", "neve", "incêndio", "incendio", "incêndios", "incendios", "bombeiros", "proteção civil", "protecao civil", "chamas", "evacuação", "evacuacao"],
    "Desporto": ["futebol", "benfica", "sporting", "fc porto", "liga dos campeões", "liga dos campeoes", "liga portuguesa", "campeonato nacional", "seleção nacional", "selecao nacional", "treinador", "jogador", "jogadores", "golo", "golos", "estádio", "estadio"],
    "Cultura": ["cultura", "cinema", "teatro", "música", "musica", "festival", "festivais", "livro", "livros", "exposição", "exposicao", "museu", "museus", "artista", "artistas", "concerto", "concertos"]
}

MONTH_MAP = {'Oct': 'October', 'Nov': 'November', 'Dec': 'December', 'Jan': 'January'}

# ==========================================
# 2. EXTRAÇÃO GERAL DA TV (TELEJORNAIS)
# ==========================================
BASE_DIR = Path("Project_Features")
ocr_files = list(BASE_DIR.glob("*_ocr.pkl"))

resultados_tv = []

for file in ocr_files:
    filename = file.stem
    if "Telejornal" not in filename:
        continue

    partes = filename.replace("_ocr", "").split("_")
    if len(partes) >= 4:
        data_formatada = f"{MONTH_MAP.get(partes[2], partes[2])}_{partes[3]}"
    else:
        continue

    try:
        df_ocr = pd.read_pickle(file)
        col_ocr = 'OCR' if 'OCR' in df_ocr.columns else ('text' if 'text' in df_ocr.columns else df_ocr.columns[-1])

        if isinstance(df_ocr[col_ocr].iloc[0], list):
            textos = [str(item.get('text', '')) for row in df_ocr[col_ocr] for item in row if isinstance(item, dict)]
            texto_completo = " ".join(textos).lower()
        else:
            texto_completo = " ".join(df_ocr[col_ocr].dropna().astype(str).tolist()).lower()

        # Contar TODOS os temas para o dia atual
        contagens_dia = {'date': data_formatada}
        for tema, keywords in THEME_ALIASES.items():
            ocorrencias = sum(len(re.findall(rf'\b{re.escape(kw)}\b', texto_completo)) for kw in keywords)
            contagens_dia[tema] = ocorrencias

        resultados_tv.append(contagens_dia)

    except Exception as e:
        pass

df_tv = pd.DataFrame(resultados_tv)

# Agrupar por data (somar frequências de dias com múltiplos telejornais)
if not df_tv.empty:
    df_tv = df_tv.groupby('date').sum().reset_index()

# ==========================================
# 3. CRUZAMENTO E MATRIZ DE PREVISÃO
# ==========================================
df_polls = pd.read_pickle('Project_Features/polls_timeline.pkl')
candidatos_cols = df_polls.columns[2:]

# Merge temporal
df_analise = pd.merge(df_polls, df_tv, on='date', how='left')

# Preencher dias sem dados de TV com 0 e remover o Gouveia e Melo (outlier matemático)
df_analise = df_analise.fillna(0)
candidatos_analise = [c for c in candidatos_cols if c != 'Gouveia_Melo']
temas_analise = list(THEME_ALIASES.keys())

# Construir a Matriz de Correlação Longitudinal
matriz_correlacao = pd.DataFrame(index=candidatos_analise, columns=temas_analise)

for candidato in candidatos_analise:
    for tema in temas_analise:
        # Correlação de Pearson entre a evolução da sondagem e a frequência do tema na TV
        corr = df_analise[candidato].corr(df_analise[tema])
        matriz_correlacao.loc[candidato, tema] = corr

# Converter para numérico
matriz_correlacao = matriz_correlacao.apply(pd.to_numeric)

# Limpar o nome das colunas de candidatos para visualização
matriz_correlacao.index = [c.replace('_', ' ') for c in matriz_correlacao.index]

# ==========================================
# 4. GERAR O HEATMAP GERAL DE AGENDA-SETTING
# ==========================================
plt.figure(figsize=(16, 8))

# Criar heatmap (Verde = TV ajudou a subir; Vermelho = TV ajudou a descer)
sns.heatmap(matriz_correlacao, annot=True, cmap='RdYlGn', fmt='.2f', vmin=-1, vmax=1,
            linewidths=0.5, cbar_kws={'label': 'Poder Preditivo Longitudinal (Correlação)'})

plt.title('Matriz de Previsão: Impacto da Agenda Mediática (TV) nas Sondagens', fontsize=16, pad=20, fontweight='bold')
plt.ylabel('Candidatos', fontsize=12, fontweight='bold')
plt.xlabel('Volume Temático nos Telejornais', fontsize=12, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)

plt.tight_layout()
plt.savefig('matriz_previsao_agenda_setting.png', dpi=300)
plt.show()

In [ ]:
import pandas as pd

# 1. Carregar as sondagens
df_polls = pd.read_pickle('Project_Features/polls_timeline.pkl')
candidatos_cols = df_polls.columns[2:] # Ignorar 'date' e 'Poll_source'

# 2. Extrair o Mês de cada sondagem
# (As datas estão no formato "October_13", extraímos a primeira parte)
df_polls['Month'] = df_polls['date'].apply(lambda x: x.split('_')[0])

# 3. Calcular a Média por Mês
# O agrupamento calcula o valor médio das várias empresas num dado mês
df_media_mensal = df_polls.groupby('Month')[candidatos_cols].mean().reset_index()

# Garantir a ordem cronológica correta
ordem_meses = ['October', 'November', 'December', 'January']
df_media_mensal['Month'] = pd.Categorical(df_media_mensal['Month'], categories=ordem_meses, ordered=True)
df_media_mensal = df_media_mensal.sort_values('Month').reset_index(drop=True)

print("--- MÉDIAS MENSAIS (Agregado Agências) ---")
print(df_media_mensal)

# 4. Calcular o Delta Limpo (Último Mês - Primeiro Mês)
deltas = []
for col in candidatos_cols:
    nome_limpo = col.replace("_", " ")
    delta_limpo = df_media_mensal[col].iloc[-1] - df_media_mensal[col].iloc[0]
    deltas.append({'candidate': nome_limpo, 'Delta_Voto': delta_limpo})

df_deltas_limpos = pd.DataFrame(deltas)

print("\n--- DELTAS FINAIS CORRIGIDOS ---")
print(df_deltas_limpos.to_string(index=False))

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# 1. DICIONÁRIO DE TEMAS (Mantém-se igual)
THEME_ALIASES = {
    "Eleições/Campanha": ["presidenciais", "presidencial", "eleições", "eleicoes", "eleição", "eleicao", "eleições presidenciais", "eleicoes presidenciais", "candidato", "candidata", "candidatos", "candidaturas", "campanha", "campanha eleitoral", "debate eleitoral", "debates eleitorais", "voto", "votos", "eleitores", "urna", "urnas"],
    "Sondagens": ["sondagem", "sondagens", "barómetro", "barometro", "intenção de voto", "intencao de voto", "intenções de voto", "intencoes de voto", "estimativa eleitoral", "projeção eleitoral", "projecao eleitoral", "pontos percentuais"],
    "Governo/Partidos": ["governo", "executivo", "primeiro ministro", "primeiro-ministro", "ministro", "ministra", "ministros", "parlamento", "assembleia da república", "assembleia da republica", "oposição", "oposicao", "líder parlamentar", "lider parlamentar", "grupo parlamentar", "maioria absoluta", "moção de censura", "mocao de censura", "partidos políticos", "partidos politicos"],
    "Saúde": ["saúde", "saude", "sns", "serviço nacional de saúde", "servico nacional de saude", "hospital", "hospitais", "médico", "medico", "médicos", "medicos", "enfermeiro", "enfermeira", "enfermeiros", "enfermeiras", "urgência", "urgencia", "urgências", "urgencias", "doente", "doentes", "utente", "utentes", "lista de espera", "listas de espera"],
    "Economia/Proteção Social": ["economia", "económico", "economico", "inflação", "inflacao", "preços", "precos", "custo de vida", "impostos", "irs", "iva", "irc", "orçamento", "orcamento", "orçamento do estado", "orcamento do estado", "défice", "defice", "dívida pública", "divida publica", "salário", "salario", "salários", "salarios", "salário mínimo", "salario minimo", "pensões", "pensoes", "reformas", "segurança social", "seguranca social", "subsídio", "subsidio", "subsídios", "subsidios", "apoios sociais", "apoio social", "empresas", "juros", "taxas de juro", "banco de portugal", "bce", "bancos"],
    "Habitação": ["habitação", "habitacao", "arrendamento", "arrendar", "renda", "rendas", "senhorio", "senhorios", "inquilino", "inquilinos", "crédito habitação", "credito habitacao", "empréstimo da casa", "emprestimo da casa", "preço das casas", "precos das casas", "mercado imobiliário", "mercado imobiliario", "imobiliário", "imobiliario"],
    "Educação": ["educação", "educacao", "escola", "escolas", "professor", "professores", "aluno", "alunos", "ensino", "aulas", "creche", "creches", "universidade", "universidades", "estudante", "estudantes", "exames nacionais", "ano letivo", "ano lectivo"],
    "Justiça/Segurança": ["justiça", "justica", "tribunal", "tribunais", "polícia", "policia", "psp", "gnr", "pj", "polícia judiciária", "policia judiciaria", "crime", "crimes", "fraude", "fraudes", "sócrates", "socrates", "josé sócrates", "jose socrates", "homicídio", "homicidio", "agressão", "agressao", "corrupção", "corrupcao", "pgr", "ministério público", "ministerio publico", "detido", "detidos", "arguido", "arguidos", "prisão", "prisao", "caução", "caucao", "buscas", "operação policial", "operacao policial", "acidente", "acidentes", "colisão", "colisao", "despiste", "queda", "descarrilamento", "vítima", "vitima", "vítimas", "vitimas", "ferido", "feridos", "ferida", "feridas", "morto", "mortos", "morta", "mortas", "falha de segurança", "falha de seguranca", "falha técnica", "falha tecnica", "investigação", "investigacao", "inquérito", "inquerito"],
    "Internacional": ["ucrânia", "ucrania", "rússia", "russia", "guerra na ucrânia", "guerra na ucrania", "israel", "gaza", "palestina", "hamas", "médio oriente", "medio oriente", "trump", "donald trump", "casa branca", "eua", "estados unidos", "bruxelas", "união europeia", "uniao europeia", "diplomacia", "diplomacia europeia", "diplomático", "diplomatico", "diplomática", "diplomatica", "relações diplomáticas", "relacoes diplomaticas", "nato", "onu", "frança", "franca", "espanha", "brasil", "china", "reino unido", "alemanha", "áfrica do sul", "africa do sul", "g20", "g7", "republicanos", "republicano", "partido republicano", "democratas", "democrata", "partido democrata", "congresso americano", "senado americano", "câmara dos representantes", "camara dos representantes"],
    "Greves/Trabalho": ["greve", "greves", "sindicato", "sindicatos", "trabalhador", "trabalhadores", "protesto", "protestos", "manifestação", "manifestacao", "manifestantes", "contrato coletivo", "contrato colectivo", "concertação social", "concertacao social"],
    #"Transportes/Mobilidade": ["transportes", "metro", "metropolitano", "comboio", "comboios", "cp", "fertagus", "autocarro", "autocarros", "trânsito", "transito", "aeroporto", "tap", "tap air portugal", "avião", "aviao", "aviões", "avioes", "estrada", "autoestrada", "portagens", "carris", "elétrico", "eletrico", "ascensor", "ascensores", "funicular", "calçada da glória", "calcada da gloria", "elevador da glória", "elevador da gloria"],
    #"Ambiente/Meteorologia/Proteção Civil": ["ambiente", "clima", "climático", "climatico", "alterações climáticas", "alteracoes climaticas", "seca", "chuva", "temporal", "inundações", "inundacoes", "cheias", "emissões", "emissoes", "poluição", "poluicao", "meteorologia", "previsão meteorológica", "previsao meteorologica", "temperatura", "vento", "frio", "calor", "neve", "incêndio", "incendio", "incêndios", "incendios", "bombeiros", "proteção civil", "protecao civil", "chamas", "evacuação", "evacuacao"],
    #"Desporto": ["futebol", "benfica", "sporting", "fc porto", "liga dos campeões", "liga dos campeoes", "liga portuguesa", "campeonato nacional", "seleção nacional", "selecao nacional", "treinador", "jogador", "jogadores", "golo", "golos", "estádio", "estadio"],
    "Cultura": ["cultura", "cinema", "teatro", "música", "musica", "festival", "festivais", "livro", "livros", "exposição", "exposicao", "museu", "museus", "artista", "artistas", "concerto", "concertos"]
}

# 2. CARREGAMENTO E FORMATAÇÃO DE DATAS
df_polls = pd.read_pickle('Project_Features/polls_timeline.pkl')
if 'Date' in df_polls.columns:
    df_polls = df_polls.rename(columns={'Date': 'date'})

candidatos_cols = [c for c in df_polls.columns if c not in ['date', 'Poll_source']]

def formatar_data(x):
    mes, dia = str(x).split('_')
    ano = '2026' if mes == 'January' else '2025'
    return pd.to_datetime(f"{mes} {dia} {ano}", format='%B %d %Y')

df_polls['date_real'] = df_polls['date'].apply(formatar_data)

# 3. EXTRAÇÃO DA TV
BASE_DIR = Path("Project_Features")
resultados_tv = []
mes_map = {'Oct': 'October', 'Nov': 'November', 'Dec': 'December', 'Jan': 'January'}

for file in BASE_DIR.glob("*_ocr.pkl"):
    if "Telejornal" not in file.stem: continue

    match = re.search(r'(October|November|December|January|Oct|Nov|Dec|Jan)_(\d+)', file.stem, re.IGNORECASE)
    if match:
        mes_ext = match.group(1).capitalize()
        data_formatada = f"{mes_map.get(mes_ext, mes_ext)}_{match.group(2)}"

        try:
            df_ocr = pd.read_pickle(file)
            col_ocr = 'OCR' if 'OCR' in df_ocr.columns else ('text' if 'text' in df_ocr.columns else df_ocr.columns[-1])
            if isinstance(df_ocr[col_ocr].iloc[0], list):
                textos = [str(item.get('text', '')) for row in df_ocr[col_ocr] for item in row if isinstance(item, dict)]
                texto_completo = " ".join(textos).lower()
            else:
                texto_completo = " ".join(df_ocr[col_ocr].dropna().astype(str).tolist()).lower()

            contagens_dia = {'date': data_formatada}
            for tema, keywords in THEME_ALIASES.items():
                contagens_dia[tema] = sum(len(re.findall(rf'\b{kw}\b', texto_completo)) for kw in keywords)
            resultados_tv.append(contagens_dia)
        except: pass

df_tv = pd.DataFrame(resultados_tv)
if not df_tv.empty:
    df_tv = df_tv.groupby('date', as_index=False).sum()
    df_tv['date_real'] = df_tv['date'].apply(formatar_data)
else:
    df_tv = pd.DataFrame(columns=['date', 'date_real'] + list(THEME_ALIASES.keys()))

# 4. A CORREÇÃO: JANELA TEMPORAL ACUMULADA (15 DIAS) E ISOLAMENTO DE AGÊNCIAS
resultados_acumulados = []

for empresa, df_empresa in df_polls.groupby('Poll_source'):
    df_empresa = df_empresa.sort_values('date_real')

    for idx, row in df_empresa.iterrows():
        data_sondagem = row['date_real']
        data_inicio = data_sondagem - pd.Timedelta(days=15) # Janela retrospectiva

        # Filtrar o volume da TV nos 15 dias ANTES da sondagem
        tv_janela = df_tv[(df_tv['date_real'] >= data_inicio) & (df_tv['date_real'] < data_sondagem)]

        linha = row.to_dict()
        for tema in THEME_ALIASES.keys():
            linha[tema] = tv_janela[tema].sum() if not tv_janela.empty else 0

        resultados_acumulados.append(linha)

df_analise = pd.DataFrame(resultados_acumulados)

# 5. CÁLCULO DA MATRIZ CORRIGIDA
candidatos_analise = candidatos_cols
matriz_final = pd.DataFrame(index=candidatos_analise, columns=THEME_ALIASES.keys(), dtype=float)

for candidato in candidatos_analise:
    for tema in THEME_ALIASES.keys():
        correlacoes_agencias = []
        for empresa, df_empresa in df_analise.groupby('Poll_source'):
            if len(df_empresa) >= 3: # Mínimo estatístico
                if df_empresa[tema].std() > 0 and df_empresa[candidato].std() > 0:
                    corr = df_empresa[tema].corr(df_empresa[candidato])
                    if not np.isnan(corr):
                        correlacoes_agencias.append(corr)

        # Média das correlações limpas
        matriz_final.loc[candidato, tema] = np.mean(correlacoes_agencias) if correlacoes_agencias else 0.0

matriz_final.index = [c.replace('_', ' ') for c in matriz_final.index]

# 6. VISUALIZAÇÃO
plt.figure(figsize=(16, 8))
sns.heatmap(matriz_final, annot=True, cmap='RdYlGn', fmt='.2f', vmin=-1, vmax=1,
            linewidths=0.5, cbar_kws={'label': 'Poder Preditivo (Janela TV 15 Dias)'})

plt.title('Matriz de Agenda-Setting Corrigida\n(Impacto da TV Acumulada nos 15 Dias Anteriores às Sondagens)',
          fontsize=15, pad=20, fontweight='bold')
plt.ylabel('Candidatos', fontsize=12, fontweight='bold')
plt.xlabel('Volume Temático nos Telejornais (Janela Acumulada)', fontsize=12, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('matriz_previsao_corrigida.png', dpi=300)
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# 1. CARREGAR E FORMATAR SONDAGENS
df_polls = pd.read_pickle('Project_Features/polls_timeline.pkl')
if 'Date' in df_polls.columns:
    df_polls = df_polls.rename(columns={'Date': 'date'})

candidatos_cols = [c for c in df_polls.columns if c not in ['date', 'Poll_source', 'date_real', 'Mes']]

def formatar_data(x):
    mes, dia = str(x).split('_')
    ano = '2026' if mes == 'January' else '2025'
    return pd.to_datetime(f"{mes} {dia} {ano}", format='%B %d %Y')

df_polls['date_real'] = df_polls['date'].apply(formatar_data)
df_polls_media = df_polls.groupby('date_real')[candidatos_cols].mean().reset_index()
df_polls_media = df_polls_media.sort_values('date_real')

# 2. PROCURAR FICHEIROS E EXTRAIR DATAS (MÚLTIPLAS PASTAS)
ficheiros_raiz = list(Path(".").glob("*_speech.pkl"))
ficheiros_features = list(Path("Project_Features").glob("*_speech.pkl"))
todos_ficheiros = ficheiros_raiz + ficheiros_features

ficheiros_debates = [f for f in todos_ficheiros if "telejornal" not in f.stem.lower()]
print(f"-> A mapear datas de {len(ficheiros_debates)} ficheiros de debates...")

mapeamento_datas = {}
mes_map = {'Oct': 'October', 'Nov': 'November', 'Dec': 'December', 'Jan': 'January'}

for file in ficheiros_debates:
    nome_ficheiro = file.stem.lower()
    match_data = re.search(r'(october|november|december|january|oct|nov|dec|jan)_(\d+)', nome_ficheiro)

    if match_data:
        mes_ext = match_data.group(1).capitalize()
        dia_ext = match_data.group(2)
        mes_formatado = mes_map.get(mes_ext, mes_ext)
        data_real = formatar_data(f"{mes_formatado}_{dia_ext}")

        # Guardar o nome do ficheiro completo para validação flexível
        mapeamento_datas[nome_ficheiro] = data_real

# 3. CRUZAR DATAS COM O CSV DE SEGMENTOS (CORRESPONDÊNCIA FLEXÍVEL)
df_segmentos = pd.read_csv('analise_temas_por_segmento.csv')
df_candidatos = df_segmentos[(df_segmentos['candidate'] != 'Moderador/Outro') &
                             (df_segmentos['Tema_Dominante'] != 'Nenhum/Não Identificado')].copy()

def associar_data_corrigida(debate_name):
    cands = str(debate_name).lower().split(' vs ')
    if len(cands) == 2:
        palavras_cand1 = cands[0].split()
        palavras_cand2 = cands[1].split()

        for nome_ficheiro, data_real in mapeamento_datas.items():
            # Confirma se todas as palavras dos dois candidatos estão contidas no nome do ficheiro
            if all(p in nome_ficheiro for p in palavras_cand1) and all(p in nome_ficheiro for p in palavras_cand2):
                return data_real
    return pd.NaT

df_candidatos['date_real'] = df_candidatos['debate_name'].apply(associar_data_corrigida)

eventos_perdidos = df_candidatos['date_real'].isna().sum()
print(f"-> Mapeamento concluído. {len(df_candidatos) - eventos_perdidos} segmentos associados. {eventos_perdidos} falhas.")
df_candidatos = df_candidatos.dropna(subset=['date_real'])

# 4. CALCULAR O IMPACTO IMEDIATO
eventos = []
for (candidato, data_debate), group in df_candidatos.groupby(['candidate', 'date_real']):
    temas_debate = (group['Tema_Dominante'].value_counts() / len(group)) * 100

    sondagens_antes = df_polls_media[df_polls_media['date_real'] <= data_debate]
    sondagens_depois = df_polls_media[df_polls_media['date_real'] > data_debate]

    if sondagens_antes.empty or sondagens_depois.empty:
        continue

    cand_formatado = candidato.replace(" ", "_")
    if cand_formatado not in df_polls_media.columns:
        continue

    voto_antes = sondagens_antes.iloc[-1][cand_formatado]
    voto_depois = sondagens_depois.iloc[0][cand_formatado]
    delta_imediato = voto_depois - voto_antes

    evento = {'candidato': candidato, 'delta_voto': delta_imediato}
    for tema, perc in temas_debate.items():
        evento[tema] = perc
    eventos.append(evento)

df_eventos = pd.DataFrame(eventos).fillna(0)

# 5. MATRIZ DE CORRELAÇÃO PREDITIVA (SPEARMAN)
if df_eventos.empty:
    print("ERRO CRÍTICO: df_eventos está vazio. Sem pontos de dados temporais para cruzar.")
else:
    peso_global_temas = df_candidatos['Tema_Dominante'].value_counts(normalize=True) * 100
    temas_relevantes = peso_global_temas[peso_global_temas >= 3.0].index

    matriz_debates = pd.DataFrame(index=candidatos_cols, columns=temas_relevantes, dtype=float)

    for candidato in candidatos_cols:
        cand_col = candidato.replace("_", " ")
        df_cand_eventos = df_eventos[df_eventos['candidato'] == cand_col]

        if len(df_cand_eventos) >= 3:
            for tema in temas_relevantes:
                if tema in df_cand_eventos.columns and df_cand_eventos[tema].std() > 0:
                    corr = df_cand_eventos[tema].corr(df_cand_eventos['delta_voto'], method='spearman')
                    matriz_debates.loc[candidato, tema] = corr if not np.isnan(corr) else 0.0
                else:
                    matriz_debates.loc[candidato, tema] = 0.0
        else:
            matriz_debates.loc[candidato] = 0.0

    matriz_debates.index = [c.replace('_', ' ') for c in matriz_debates.index]

    # 6. GERAR GRÁFICO FINAL
    plt.figure(figsize=(14, 8))
    sns.heatmap(matriz_debates, annot=True, cmap='RdYlGn', fmt='.2f', vmin=-1, vmax=1,
                linewidths=0.5, cbar_kws={'label': 'Poder Preditivo Imediato (Spearman)'})

    plt.title('Matriz de Impacto Tático nos Debates\n(Correlação Spearman entre Foco Temático e Sondagens)',
              fontsize=15, pad=20, fontweight='bold')
    plt.ylabel('Candidatos', fontsize=12, fontweight='bold')
    plt.xlabel('Eixos Estruturais nos Debates', fontsize=12, fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)

    plt.tight_layout()
    plt.savefig('matriz_previsao_debates_dinamica.png', dpi=300)
    plt.show()
    print("-> Matriz gerada com sucesso.")

TESTES EMBEDDINGS

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
from pathlib import Path

# 1. DEFINE PATHS
pasta = "Project_Features"
# Use glob to find all speech files as you did in your notebook
ficheiros_speech = glob.glob(os.path.join(pasta, "*_speech.pkl"))

print(f"Loading {len(ficheiros_speech)} speech files to extract text embeddings...\n")

lista_dfs = []

# 2. LOAD AND PROCESS EMBEDDINGS
for caminho in ficheiros_speech:
    nome_ficheiro = os.path.basename(caminho).replace('_speech.pkl', '')

    try:
        # Load the pickle file[cite: 2]
        df_temp = pd.read_pickle(caminho)

        # Skip empty files
        if len(df_temp) == 0:
            continue

        # Add the video_id so we know where this came from[cite: 2]
        df_temp['video_id'] = nome_ficheiro

        # The crucial part: Ensure text_embedding is a usable numpy array
        # Some loaders might bring them in as strings or lists. We want a 2D matrix eventually.
        if 'text_embedding' in df_temp.columns:
            # Drop rows where the embedding might be missing
            df_temp = df_temp.dropna(subset=['text_embedding'])

            # Convert the embedding column to a list of numpy arrays
            # This makes it much easier to stack them later for machine learning
            df_temp['text_embedding'] = df_temp['text_embedding'].apply(lambda x: np.array(x, dtype=float))

            lista_dfs.append(df_temp)
        else:
            print(f"Aviso: Ficheiro {nome_ficheiro} não tem a coluna 'text_embedding'.")

    except Exception as e:
        print(f"Erro no ficheiro {nome_ficheiro}: {e}")

# 3. COMBINE INTO FINAL DATASET
if lista_dfs:
    df_principal = pd.concat(lista_dfs, ignore_index=True)
    print(f"\\nDataFrame principal criado com sucesso! Tem {len(df_principal)} segmentos.")

    # Show the structure to confirm the embedding is there
    display(df_principal[['video_id', 'transcript', 'text_embedding']].head())

    # Example: How to convert the embedding column into a proper Matrix for ML
    # If you have 1000 rows and embeddings of size 384, X will be shape (1000, 384)
    X_embeddings = np.stack(df_principal['text_embedding'].values)
    print(f"\\nFormato da Matriz de Embeddings pronta para Análise: {X_embeddings.shape}")
else:
    print("Aviso: Nenhum dado carregado.")

In [ ]:
import pandas as pd
import numpy as np
import glob
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from sklearn.feature_extraction.text import CountVectorizer

# 1. CARREGAR OS DADOS E EMBEDDINGS
pasta = "Project_Features"
ficheiros_speech = glob.glob(os.path.join(pasta, "*_speech.pkl"))

lista_dfs = []
for caminho in ficheiros_speech:
    try:
        df_temp = pd.read_pickle(caminho)
        if len(df_temp) > 0 and 'text_embedding' in df_temp.columns:
            df_temp = df_temp.dropna(subset=['text_embedding'])
            df_temp['text_embedding'] = df_temp['text_embedding'].apply(lambda x: np.array(x, dtype=float))
            df_temp['video_id'] = os.path.basename(caminho).replace('_speech.pkl', '')
            lista_dfs.append(df_temp)
    except:
        pass

df_principal = pd.concat(lista_dfs, ignore_index=True)

# Criar a Matriz X de embeddings
X_embeddings = np.stack(df_principal['text_embedding'].values)
print(f"Matriz de Embeddings criada com a dimensão: {X_embeddings.shape}")

# 2. APLICAR CLUSTERING (K-MEANS)
# Vamos tentar encontrar 8 temas distintos (podes ajustar este número depois)
NUM_CLUSTERS = 15
kmeans = KMeans(n_clusters=NUM_CLUSTERS, random_state=42, n_init=10)
df_principal['Cluster'] = kmeans.fit_predict(X_embeddings)

# 3. VISUALIZAÇÃO COM t-SNE (Redução de dimensionalidade para 2D)
print("A calcular o t-SNE para visualização (isto pode demorar alguns segundos)...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
embeddings_2d = tsne.fit_transform(X_embeddings)

df_principal['tsne_x'] = embeddings_2d[:, 0]
df_principal['tsne_y'] = embeddings_2d[:, 1]

plt.figure(figsize=(12, 8))
sns.scatterplot(
    data=df_principal,
    x='tsne_x',
    y='tsne_y',
    hue='Cluster',
    palette='tab10',
    alpha=0.7,
    edgecolor=None
)
plt.title("Mapeamento Semântico dos Discursos (Clusters de Embeddings)", fontsize=16, fontweight='bold')
plt.xlabel("Dimensão t-SNE 1")
plt.ylabel("Dimensão t-SNE 2")
plt.legend(title='Tema (Cluster)', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

# 4. DESCOBRIR O SIGNIFICADO DE CADA CLUSTER
print("\n--- PALAVRAS-CHAVE POR CLUSTER (TEMA) ---")
stopwords_pt = ['que', 'de', 'não', 'a', 'o', 'e', 'é', 'um', 'uma', 'os', 'as', 'para', 'com', 'por', 'do', 'da', 'em', 'no', 'na', 'se', 'mas', 'como', 'mais', 'ou', 'eu', 'nós', 'ele', 'tem', 'isso', 'este', 'esta', 'são', 'ter', 'ser', 'foi', 'ao', 'dos', 'das', 'sua', 'seu', 'já', 'essa', 'esse', 'porque', 'está', 'temos', 'vai', 'seja', 'há', 'quando', 'fazer', 'só', 'acho', 'faz', 'vão', 'muito', 'coisa', 'estou', 'então', 'também', 'nos', 'me', 'aos', 'quem', 'aos', 'era', 'quem', 'nem', 'tudo', 'agora', 'aqui', 'sobre', 'até', 'mesmo', 'onde', 'qual', 'pelo']

vectorizer = CountVectorizer(stop_words=stopwords_pt, max_features=10)

for cluster_num in range(NUM_CLUSTERS):
    textos_cluster = df_principal[df_principal['Cluster'] == cluster_num]['transcript'].astype(str)

    if len(textos_cluster) > 0:
        matriz_palavras = vectorizer.fit_transform(textos_cluster)
        palavras_frequentes = vectorizer.get_feature_names_out()

        print(f"Cluster {cluster_num} ({len(textos_cluster)} segmentos):")
        print(f"  Palavras Top: {', '.join(palavras_frequentes)}")
        print("-" * 50)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

# 1. NORMALIZAR EMBEDDINGS
X_normalized = normalize(X_embeddings)

# 2. RE-CALCULAR CLUSTERS
NUM_CLUSTERS = 15
kmeans = KMeans(n_clusters=NUM_CLUSTERS, random_state=42, n_init=20)
df_principal['Cluster'] = kmeans.fit_predict(X_normalized)

# 3. LISTA DE STOPWORDS
stopwords_pt_avancadas = [
    'que', 'de', 'não', 'a', 'o', 'e', 'é', 'um', 'uma', 'os', 'as', 'para', 'com', 'por', 'do', 'da',
    'em', 'no', 'na', 'se', 'mas', 'como', 'mais', 'ou', 'eu', 'nós', 'ele', 'tem', 'isso', 'este', 'esta',
    'são', 'ter', 'ser', 'foi', 'ao', 'dos', 'das', 'sua', 'seu', 'já', 'essa', 'esse', 'porque', 'está',
    'temos', 'vai', 'seja', 'há', 'quando', 'fazer', 'só', 'acho', 'faz', 'vão', 'muito', 'coisa', 'estou',
    'então', 'também', 'nos', 'me', 'aos', 'quem', 'era', 'nem', 'tudo', 'agora', 'aqui', 'sobre', 'até',
    'mesmo', 'onde', 'qual', 'pelo', 'dizer', 'quer', 'quero', 'sei', 'senhor', 'sou', 'tenho', 'vou',
    'estão', 'portanto', 'todos', 'anos', 'bem', 'depois', 'entre', 'hoje', 'noite', 'aquilo', 'minha',
    'pode', 'ainda', 'dia', 'foram', 'nas', 'estes', 'mim', 'ver', 'têm', 'deu', 'ria', 'país', 'portugal',
    'governo', 'presidente', 'república', 'pessoas', 'primeiro', 'ministro', 'candidato', 'candidatos',
    'fala', 'falou', 'sempre', 'pode', 'podemos', 'neste', 'nesta', 'dois', 'duas', 'parte', 'vez', 'vezes'
]

# 4. EXTRAÇÃO DE TEMAS COM TF-IDF (Destaca o que é ÚNICO em cada cluster)
print("\n--- PALAVRAS-CHAVE POR CLUSTER (TF-IDF AFINADO) ---")

for cluster_num in range(NUM_CLUSTERS):
    textos_cluster = df_principal[df_principal['Cluster'] == cluster_num]['transcript'].astype(str)

    if len(textos_cluster) > 0:
        # Instanciar um TF-IDF limpo para este grupo
        tfidf = TfidfVectorizer(stop_words=stopwords_pt_avancadas, max_features=1000)
        matriz_tfidf = tfidf.fit_transform(textos_cluster)

        # Somar a "importância" de cada palavra
        scores_soma = matriz_tfidf.sum(axis=0).A1
        palavras = tfidf.get_feature_names_out()

        # Obter o top 10 palavras mais distintivas
        top_indices = scores_soma.argsort()[-10:][::-1]
        top_palavras = [palavras[i] for i in top_indices]

        print(f"Cluster {cluster_num} ({len(textos_cluster)} segmentos):")
        print(f"  Temas Fortes: {', '.join(top_palavras)}")
        print("-" * 50)

In [ ]:
# 1. EXPANDIR A LISTA DE STOPWORDS (As anteriores + novas descobertas)
stopwords_pt_finais = stopwords_pt_avancadas + [
    'estado', 'gente', 'isto', 'vamos', 'assim', 'tempo', 'sabe', 'falar',
    'disse', 'estava', 'deve', 'nossa', 'nosso', 'nunca', 'claro', 'forma',
    'estar', 'te', 'lá', 'menos', 'momento', 'estamos', 'preciso', 'às',
    'apenas', 'cerca', 'três', 'ano', 'ela', 'ele', 'delas', 'deles', 'sim',
    'naquela', 'naquele', 'dar', 'dado', 'toda', 'todas', 'seus', 'suas',
    'tão', 'vem', 'vêm', 'faz', 'feito', 'quais', 'cada', 'qualquer', 'alguns',
    'algumas', 'outro', 'outra', 'outros', 'outras', 'tinha', 'tinham',
    'teve', 'tiveram', 'fosse', 'fossem', 'debate', 'frente', 'rtp', 'joão',
    'antónio', 'marcos', 'nós', 'lhe', 'diz', 'aí', 'ali'
]

# 2. RE-EXTRAIR OS TEMAS COM O NOVO FILTRO
print("\n--- PALAVRAS-CHAVE POR CLUSTER (FILTRO FINAL) ---")

for cluster_num in range(NUM_CLUSTERS):
    textos_cluster = df_principal[df_principal['Cluster'] == cluster_num]['transcript'].astype(str)

    if len(textos_cluster) > 0:
        tfidf = TfidfVectorizer(stop_words=stopwords_pt_finais, max_features=1000)
        matriz_tfidf = tfidf.fit_transform(textos_cluster)

        scores_soma = matriz_tfidf.sum(axis=0).A1
        palavras = tfidf.get_feature_names_out()

        top_indices = scores_soma.argsort()[-10:][::-1]
        top_palavras = [palavras[i] for i in top_indices]

        print(f"Cluster {cluster_num} ({len(textos_cluster)} segmentos):")
        print(f"  Temas Fortes: {', '.join(top_palavras)}")
        print("-" * 50)

In [ ]:
# 1. STOPWORDS
stopwords_pt_finais = stopwords_pt_avancadas + [
    'estado', 'gente', 'isto', 'vamos', 'assim', 'tempo', 'sabe', 'falar',
    'disse', 'estava', 'deve', 'nossa', 'nosso', 'nunca', 'claro', 'forma',
    'estar', 'te', 'lá', 'menos', 'momento', 'estamos', 'preciso', 'às',
    'apenas', 'cerca', 'três', 'ano', 'ela', 'ele', 'delas', 'deles', 'sim',
    'naquela', 'naquele', 'dar', 'dado', 'toda', 'todas', 'seus', 'suas',
    'tão', 'vem', 'vêm', 'faz', 'feito', 'quais', 'cada', 'qualquer', 'alguns',
    'algumas', 'outro', 'outra', 'outros', 'outras', 'tinha', 'tinham',
    'teve', 'tiveram', 'fosse', 'fossem', 'debate', 'frente', 'rtp', 'joão',
    'antónio', 'marcos', 'nós', 'lhe', 'diz', 'aí', 'ali',
    # NOVAS ADIÇÕES:
    'exemplo', 'verdade', 'você', 'lado', 'meu', 'nada', 'questão', 'relação',
    'importante', 'vida', 'pessoa', 'quase', 'número', 'semana', 'boa', 'dias',
    'podem', 'pouco', 'sem', 'meio', 'ter', 'sido', 'ver', 'certeza', 'parte'
]

# 2. RE-EXTRAIR OS TEMAS COM O FILTRO FINAL
print("\n--- PALAVRAS-CHAVE POR CLUSTER ---")

for cluster_num in range(NUM_CLUSTERS):
    textos_cluster = df_principal[df_principal['Cluster'] == cluster_num]['transcript'].astype(str)

    if len(textos_cluster) > 0:
        tfidf = TfidfVectorizer(stop_words=stopwords_pt_finais, max_features=1000)
        matriz_tfidf = tfidf.fit_transform(textos_cluster)

        scores_soma = matriz_tfidf.sum(axis=0).A1
        palavras = tfidf.get_feature_names_out()

        top_indices = scores_soma.argsort()[-10:][::-1]
        top_palavras = [palavras[i] for i in top_indices]

        print(f"Cluster {cluster_num} ({len(textos_cluster)} segmentos):")
        print(f"  Temas Fortes: {', '.join(top_palavras)}")
        print("-" * 50)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# ==========================================
# 1. MAPEAMENTO E LIMPEZA DE TEMAS
# ==========================================
# Ajusta os rótulos consoante o output da tua execução do TF-IDF
mapa_temas = {
    0: "Campanha/Presidenciais",
    1: "Economia/Trabalho",
    2: "Direitos Laborais/Sociais",
    3: "Europa/Constituição",
    4: "Retórica de Ataque",
    5: "Lixo/Alucinações", # Será excluído
    6: "Internacional",
    7: "Orçamento/Finanças/Saúde"
}

df_principal['Nome_Tema'] = df_principal['Cluster'].map(mapa_temas)

# Excluir os segmentos identificados como erros do Whisper
df_valido = df_principal[df_principal['Cluster'] != 5].copy()

# ==========================================
# 2. CÁLCULO DAS PERCENTAGENS DE TEMPO
# ==========================================
# Somar a duração por tema em cada vídeo
tempo_tema = df_valido.groupby(['video_id', 'Nome_Tema'])['duration'].sum().reset_index()

# Somar o tempo útil total de cada vídeo
tempo_total = df_valido.groupby('video_id')['duration'].sum().reset_index().rename(columns={'duration': 'duration_total'})

# Cruzar e calcular a percentagem
df_plot = pd.merge(tempo_tema, tempo_total, on='video_id')
df_plot['Percentagem'] = (df_plot['duration'] / df_plot['duration_total']) * 100

# ==========================================
# 3. MATRIZ DE VISUALIZAÇÃO E GRÁFICO
# ==========================================
# Reestruturar os dados para barras empilhadas
df_pivot = df_plot.pivot(index='video_id', columns='Nome_Tema', values='Percentagem').fillna(0)

# Gerar gráfico
ax = df_pivot.plot(kind='bar', stacked=True, figsize=(16, 8), colormap='tab10', edgecolor='white', linewidth=0.5)

plt.title('Distribuição Semântica por Debate/Telejornal (% de Tempo Útil)', fontsize=15, fontweight='bold', pad=15)
plt.xlabel('Evento', fontweight='bold', fontsize=12)
plt.ylabel('Tempo Dedicado (%)', fontweight='bold', fontsize=12)

# Colocar a legenda fora do gráfico
plt.legend(title='Temas (TF-IDF)', bbox_to_anchor=(1.02, 1), loc='upper left')

plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('distribuicao_temas_percentagem.png', dpi=300)
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Colar aqui o THEME_ALIASES otimizado (o bloco de código acima)

# =======================================================
# 1. CARREGAR E FORMATAR SONDAGENS
# =======================================================
df_polls = pd.read_pickle('Project_Features/polls_timeline.pkl')
if 'Date' in df_polls.columns:
    df_polls = df_polls.rename(columns={'Date': 'date'})

candidatos_cols = [c for c in df_polls.columns if c not in ['date', 'Poll_source', 'date_real', 'Mes']]

def formatar_data(x):
    mes, dia = str(x).split('_')
    ano = '2026' if mes == 'January' else '2025'
    return pd.to_datetime(f"{mes} {dia} {ano}", format='%B %d %Y')

df_polls['date_real'] = df_polls['date'].apply(formatar_data)
df_polls_media = df_polls.groupby('date_real')[candidatos_cols].mean().reset_index()
df_polls_media = df_polls_media.sort_values('date_real')

# =======================================================
# 2. PROCURAR FICHEIROS E EXTRAIR DATAS
# =======================================================
ficheiros_raiz = list(Path(".").glob("*_speech.pkl"))
ficheiros_features = list(Path("Project_Features").glob("*_speech.pkl"))
todos_ficheiros = ficheiros_raiz + ficheiros_features

ficheiros_debates = [f for f in todos_ficheiros if "telejornal" not in f.stem.lower()]

mapeamento_datas = {}
mes_map = {'Oct': 'October', 'Nov': 'November', 'Dec': 'December', 'Jan': 'January'}

for file in ficheiros_debates:
    nome_ficheiro = file.stem.lower()
    match_data = re.search(r'(october|november|december|january|oct|nov|dec|jan)_(\d+)', nome_ficheiro)

    if match_data:
        mes_ext = match_data.group(1).capitalize()
        dia_ext = match_data.group(2)
        mes_formatado = mes_map.get(mes_ext, mes_ext)
        data_real = formatar_data(f"{mes_formatado}_{dia_ext}")
        mapeamento_datas[nome_ficheiro] = data_real

# =======================================================
# 3. EXTRAÇÃO MULTI-LABEL COM O DICIONÁRIO OTIMIZADO
# =======================================================
df_segmentos = pd.read_excel('analise_temas_por_segmento.xlsx')
df_candidatos = df_segmentos[df_segmentos['candidate'] != 'Moderador/Outro'].copy()

def associar_data_corrigida(debate_name):
    cands = str(debate_name).lower().split(' vs ')
    if len(cands) == 2:
        palavras_cand1 = cands[0].split()
        palavras_cand2 = cands[1].split()
        for nome_ficheiro, data_real in mapeamento_datas.items():
            if all(p in nome_ficheiro for p in palavras_cand1) and all(p in nome_ficheiro for p in palavras_cand2):
                return data_real
    return pd.NaT

df_candidatos['date_real'] = df_candidatos['debate_name'].apply(associar_data_corrigida)
df_candidatos = df_candidatos.dropna(subset=['date_real'])

# Contar as palavras do dicionário em cada segmento
for tema, keywords in THEME_ALIASES.items():
    pattern = r'\b(' + '|'.join(keywords) + r')\b'
    df_candidatos[tema] = df_candidatos['transcript'].astype(str).str.count(pattern, flags=re.IGNORECASE)

# =======================================================
# 4. CALCULAR A EFICÁCIA TÁTICA E O IMPACTO (DELTA)
# ==========================================
eventos = []
lista_temas = list(THEME_ALIASES.keys())

for (candidato, data_debate), group in df_candidatos.groupby(['candidate', 'date_real']):

    # Calcular o Perfil Tático (% de vezes que o candidato invocou cada tema neste debate)
    soma_hits = group[lista_temas].sum().sum()
    if soma_hits == 0: continue # Ignorar se não disse nenhuma keyword

    perfil_tatico = (group[lista_temas].sum() / soma_hits) * 100

    # Obter sondagens
    sondagens_antes = df_polls_media[df_polls_media['date_real'] <= data_debate]
    sondagens_depois = df_polls_media[df_polls_media['date_real'] > data_debate]

    if sondagens_antes.empty or sondagens_depois.empty: continue

    cand_formatado = candidato.replace(" ", "_")
    if cand_formatado not in df_polls_media.columns: continue

    voto_antes = sondagens_antes.iloc[-1][cand_formatado]
    voto_depois = sondagens_depois.iloc[0][cand_formatado]
    delta_imediato = voto_depois - voto_antes

    evento = {'candidato': candidato, 'delta_voto': delta_imediato}
    for tema in lista_temas:
        evento[tema] = perfil_tatico[tema]
    eventos.append(evento)

df_eventos = pd.DataFrame(eventos).fillna(0)

# =======================================================
# 5. MATRIZ DE CORRELAÇÃO PREDITIVA (SPEARMAN)
# =======================================================
matriz_debates = pd.DataFrame(index=candidatos_cols, columns=lista_temas, dtype=float)

for candidato in candidatos_cols:
    cand_col = candidato.replace("_", " ")
    df_cand_eventos = df_eventos[df_eventos['candidato'] == cand_col]

    if len(df_cand_eventos) >= 3: # Validade estatística
        for tema in lista_temas:
            if df_cand_eventos[tema].std() > 0:
                corr = df_cand_eventos[tema].corr(df_cand_eventos['delta_voto'], method='spearman')
                matriz_debates.loc[candidato, tema] = corr if not np.isnan(corr) else 0.0
            else:
                matriz_debates.loc[candidato, tema] = 0.0
    else:
        matriz_debates.loc[candidato] = 0.0

matriz_debates.index = [c.replace('_', ' ') for c in matriz_debates.index]

# =======================================================
# 6. GERAR HEATMAP FINAL
# =======================================================
# Ocultar colunas vazias (temas residuais como Desporto que ninguém falou)
matriz_limpa = matriz_debates.loc[:, (matriz_debates != 0).any(axis=0)]

plt.figure(figsize=(16, 9))
sns.heatmap(matriz_limpa, annot=True, cmap='RdYlGn', fmt='.2f', vmin=-1, vmax=1,
            linewidths=0.5, cbar_kws={'label': 'Poder Preditivo Imediato (Spearman)'})

plt.title('Matriz de Previsão de Debates Definitiva\n(Dicionário Otimizado via Machine Learning)',
          fontsize=16, pad=20, fontweight='bold')
plt.ylabel('Candidatos', fontsize=12, fontweight='bold')
plt.xlabel('Eixos Estruturais de Campanha', fontsize=12, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)

plt.tight_layout()
plt.savefig('matriz_previsao_debates_final.png', dpi=300)
plt.show()